# MODNet portrait matting — DIMER E2E matting fine-tuning tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/modnet-matting-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/modnet-matting-pipeline/blob/main/tutorials/modnet_matting_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-XM5354%2FModnet__models-ffcc4d?style=flat)](https://huggingface.co/XM5354/Modnet_models) [![Upstream](https://img.shields.io/badge/Upstream-ZHKKKe%2FMODNet-181717?style=flat&logo=github&logoColor=white)](https://github.com/ZHKKKe/MODNet) [![Paper](https://img.shields.io/badge/arXiv-2011.11961-b31b1b.svg)](https://arxiv.org/abs/2011.11961)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** trimap-free portrait alpha matting with MODNet (MobileNetV2 semantic branch, detail branch, fusion branch), held-out MAD / MSE / SAD against constant baselines, and bounded fine-tuning of the matting branches to labelled portrait/alpha pairs

**This notebook is standalone.** It carries the repository's package (4 modules under `src/modnet_matting_pipeline/`, at revision `71afecdd0f0f`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `71aca6d04ed0267b4b12bde776868f2b9fb1d06f` (~26 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh runtime (CPU or GPU) installs the pinned dependencies (torch, numpy, Pillow, safetensors, huggingface-hub), stages and digest-verifies the pinned MODNet checkpoint (25 MB) from the Hub, statically audits the legacy torch pickle against an allow-list, converts it once into safetensors with a pinned digest, builds the vendored architecture and loads it strictly, renders 80 synthetic portraits with exact alpha mattes in the kernel (48 training, 12 validation, 20 test), fetches four digest-pinned CC0 photographs (9.3 MB, no credential), scores the frozen model against the all-background and all-foreground baselines and mattes the photographs, runs a bounded fine-tuning of the matting branches, scores the same held-out portraits again, mattes the photographs again, exports the adapter as safetensors with a manifest, and reloads that artifact into a fresh pipeline to verify prediction parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5). On a T4 the model time is under a minute; on a CPU the adaptation takes a few minutes.

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4 and re-run from that cell to supply your own labelled portraits as a zip holding `pairs.csv` (columns `id`, `image`, `alpha`) beside RGB images and 8-bit greyscale alpha PNGs (0 = background, 255 = subject); at least four pairs. Pairs are resized to 512 × 512, split by seed into training, validation and test sets and flow through the same contract — validation, frozen baseline, adaptation, held-out evaluation, inference, artifact export and reload parity. The expected schema, the ceilings and the privacy guidance are stated in the Prerequisites and in Section 4, and uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

MODNet (Ke et al., AAAI 2022) predicts a portrait's alpha matte from an RGB image alone — no trimap — with three branches trained together: a MobileNetV2 low-resolution branch for the coarse semantics, a high-resolution branch for the boundary detail, and a fusion branch that produces the matte. The photographic checkpoint packaged here is the authors' release, published on Google Drive; the byte-identical file is mirrored on the Hugging Face Hub, which is what Section 3 pins at an immutable revision, with the Drive digest recorded beside it.

Two things about this row are handled in the open. **The upstream asset is a pickle** — a legacy `torch.save` file made of five pickle streams followed by raw tensor bytes. Section 3 downloads and digest-verifies it, statically lists every global those streams would import (a state dict of tensors and nothing else), refuses anything outside that allow-list, unpickles it exactly once through torch's weights-only loader, and writes a safetensors file whose digest is pinned in the carried module; the model you run is the architecture vendored in the carried `modeling.py` and loads that file strictly. **The labelled portraits are drawn, not photographed**: no portrait-matting dataset with per-pixel alpha mattes is both permissively licensed and free of personal-data concerns, so Section 4 renders figures — head, shoulders, a hair cap and dozens of hair strands with fractional coverage — over generated backgrounds with an exact alpha. They are out of the photographic training domain on purpose: the frozen model's error on them, the adapted model's error, and the model's behaviour on four real CC0 photographs before and after adaptation are the tutorial's evidence; the point of the contract is the same recipe on *your* labelled portraits.

**Learning objectives:** install the pinned runtime; inspect the carried pipeline, dataset, metrics and model modules; stage and digest-verify a legacy pickled checkpoint, read its static audit and see it converted into safetensors; render labelled portraits with exact mattes and validate them with refusal probes; read MAD, MSE, SAD and the unknown-band MAD against constant baselines; run a bounded fine-tuning with the upstream semantic / detail / matte losses, explicit hyperparameters and frozen BatchNorm statistics; compare the adapted and frozen models on the same held-out portraits and on real photographs; and export a safetensors adapter that reloads against the pinned base with verified parity.

**This notebook does not demonstrate:** video matting, the self-supervised SOC adaptation of the paper, trimap-based matting, background replacement quality beyond a simple composite, the published PPM-100 benchmark scores, face detection or recognition, and any claim that 20 drawn portraits stand in for an evaluation on photographs. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime — Google Colab (CPU or T4), Kaggle, or a Jupyter kernel with Python 3.12. The model has 6.5 M parameters: a 512 × 512 matte takes well under a second on a CPU, the default adaptation about 2 s per epoch on a T4 and about half a minute per epoch on a laptop CPU. About 200 MB of disk is needed for the checkpoint, its conversion and the photographs.
- **Knowledge:** what an alpha matte is (per-pixel opacity of the subject, fractional along hair and soft edges), what a trimap's unknown band is, and how MAD / MSE / SAD are read against a constant baseline.
- **Executable serialization handled explicitly:** the pinned checkpoint is a legacy torch pickle. It is digest-verified, statically audited against an allow-list (audit digest pinned) and unpickled **once** through torch's weights-only loader to produce the safetensors the model is actually loaded from. No Hub-hosted Python module is imported; the architecture is carried verbatim from the repository (`modeling.py`, vendored from the upstream repository at a pinned commit).
- **Data contract:** a record is `{{id, image, alpha}}` — an RGB uint8 image (any size with both sides in [64, 4096] for inference; exactly 512 × 512 for labelled records) and an alpha in [0, 1] of the same size. Validation is structural: nothing checks that the image shows a person, that the alpha belongs to the image, or that the alpha marks the subject rather than something else.
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there — photographs of identifiable people you have no consent to process are exactly that. The default path uploads nothing; its four photographs are CC0 stock portraits.
- **External access (data):** besides the model snapshot, the default path fetches four pinned objects over HTTPS — CC0 portrait photographs from Wikimedia Commons (Pixabay uploads; 9.3 MB in total), each refused on a size or SHA-256 mismatch. The labelled portraits are rendered in the kernel and need no download.
- **External access:** the Hugging Face Hub only, to fetch the pinned `XM5354/Modnet_models` snapshot (~26 MB in total) at revision `71aca6d04ed0…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `PIL` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'safetensors==0.8.0',
    'huggingface-hub==1.32.0',
]
NOTEBOOK_SOURCE = {
    'repository': 'modnet-matting-pipeline',
    'repository_revision': '71afecdd0f0f0f2b71541cd1d0a10d0eaf100c1e',
    'embedded_module': 'src/modnet_matting_pipeline/pipeline.py',
    'embedded_modules': ['src/modnet_matting_pipeline/metrics.py', 'src/modnet_matting_pipeline/modeling.py', 'src/modnet_matting_pipeline/pipeline.py', 'src/modnet_matting_pipeline/samples.py'],
    'module_sha256': '2f7238de5cded7a59c8a8b818304e3ab836a7d4b7b2c6bd54046dfe6034c4251',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, PIL
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'PIL': PIL.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/modnet_matting_pipeline/` @ `71afecdd0f0f`)

The next 4 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (2 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/4:** `src/modnet_matting_pipeline/metrics.py`

In [ ]:
"""Alpha-matte metrics (numpy only): MAD, MSE and SAD over whole mattes, MAD over the trimap's unknown band, and
the two constant baselines (all background, all foreground) scored on the same pixels."""

from __future__ import annotations

from collections.abc import Sequence
from typing import Any


def matte_errors(predicted: Any, reference: Any, *, unknown: Any | None = None) -> dict[str, float]:
    """Errors of one predicted alpha matte against its reference (both (H, W) float in [0, 1]).

    `mad` and `mse` are means over every pixel; `sad` is the summed absolute difference divided by 1,000 (the
    matting-literature convention); `mad_unknown` is the MAD over the pixels of the trimap's unknown band when
    `unknown` (a boolean (H, W) mask) is given and non-empty, else None.
    """
    import numpy as np

    p = np.asarray(predicted, dtype=np.float32)
    r = np.asarray(reference, dtype=np.float32)
    if p.shape != r.shape or p.ndim != 2:
        raise ValueError(f"predicted and reference mattes must share one (H, W) shape, got {p.shape} and {r.shape}")
    diff = np.abs(p - r)
    out = {
        "mad": float(diff.mean()),
        "mse": float(np.square(p - r).mean()),
        "sad": float(diff.sum() / 1000.0),
    }
    if unknown is not None:
        band = np.asarray(unknown, dtype=bool)
        if band.shape != p.shape:
            raise ValueError(f"unknown mask must have shape {p.shape}, got {band.shape}")
        out["mad_unknown"] = float(diff[band].mean()) if band.any() else None
    else:
        out["mad_unknown"] = None
    return out


def _mean_of(values: Sequence[float | None]) -> float | None:
    present = [v for v in values if v is not None]
    return float(sum(present) / len(present)) if present else None


def matting_metrics(
    predicted: Sequence[Any], reference: Sequence[Any], *, unknown: Sequence[Any] | None = None
) -> dict[str, Any]:
    """Per-image errors averaged over a set of mattes (each image weighs the same, whatever its size)."""
    if len(predicted) != len(reference) or not predicted:
        raise ValueError("predicted and reference must be non-empty sequences of equal length")
    if unknown is not None and len(unknown) != len(predicted):
        raise ValueError("unknown must have one mask per matte")
    rows = [
        matte_errors(p, r, unknown=None if unknown is None else unknown[i])
        for i, (p, r) in enumerate(zip(predicted, reference, strict=True))
    ]
    return {
        "n_images": len(rows),
        "mad": round(_mean_of([row["mad"] for row in rows]), 6),
        "mse": round(_mean_of([row["mse"] for row in rows]), 6),
        "sad": round(_mean_of([row["sad"] for row in rows]), 4),
        "mad_unknown": (lambda v: None if v is None else round(v, 6))(_mean_of([row["mad_unknown"] for row in rows])),
        "per_image": [{k: (None if v is None else round(v, 6)) for k, v in row.items()} for row in rows],
    }


def constant_baselines(reference: Sequence[Any], *, unknown: Sequence[Any] | None = None) -> dict[str, dict[str, Any]]:
    """The all-background (alpha 0) and all-foreground (alpha 1) mattes scored on the same references: what a
    model that never separates the subject from the scene would score."""
    import numpy as np

    out = {}
    for name, value in (("all_background", 0.0), ("all_foreground", 1.0)):
        constants = [np.full(np.asarray(r).shape, value, dtype=np.float32) for r in reference]
        metrics = matting_metrics(constants, reference, unknown=unknown)
        out[name] = {k: v for k, v in metrics.items() if k != "per_image"}
    return out

**Module 2/4:** `src/modnet_matting_pipeline/modeling.py` (carried verbatim; see the note above)

In [ ]:
"""MODNet architecture (Ke et al., 2022), vendored from `ZHKKKe/MODNet` at commit
`28165a451e4610c9d77cfdf925a94610bb2810fb` (Apache-2.0; `src/models/modnet.py` SHA-256 `2f26f5f0…`,
`src/models/backbones/mobilenetv2.py` `e3cc8ad6…`, `src/models/backbones/wrapper.py` `41197be7…`).

Parameter and buffer names are the upstream ones, so the pinned checkpoint loads strictly once its `module.`
(DataParallel) prefix is stripped. Differences from upstream, all deliberate: no `torch.load` of a backbone
checkpoint (`backbone_pretrained` is gone — the pinned matting checkpoint carries the backbone), no
`exit()` calls, no `nn.DataParallel`, `forward(img, inference)` unchanged in signature and semantics. Nothing
here imports outside `torch`, and the module is imported lazily by `pipeline.py` (fleet RTM-001).
"""

from __future__ import annotations

import torch
import torch.nn.functional as F
from torch import nn

# --------------------------------------------------------------------------------------------------
# MobileNetV2 backbone (upstream: adapted from thuyngch/Human-Segmentation-PyTorch)
# --------------------------------------------------------------------------------------------------


def _make_divisible(v: float, divisor: int, min_value: int | None = None) -> int:
    if min_value is None:
        min_value = divisor
    new_v = max(min_value, int(v + divisor / 2) // divisor * divisor)
    if new_v < 0.9 * v:  # make sure that rounding down does not go down by more than 10 %
        new_v += divisor
    return new_v


def _conv_bn(inp: int, oup: int, stride: int) -> nn.Sequential:
    return nn.Sequential(nn.Conv2d(inp, oup, 3, stride, 1, bias=False), nn.BatchNorm2d(oup), nn.ReLU6(inplace=True))


def _conv_1x1_bn(inp: int, oup: int) -> nn.Sequential:
    return nn.Sequential(nn.Conv2d(inp, oup, 1, 1, 0, bias=False), nn.BatchNorm2d(oup), nn.ReLU6(inplace=True))


class InvertedResidual(nn.Module):
    def __init__(self, inp: int, oup: int, stride: int, expansion: int, dilation: int = 1) -> None:
        super().__init__()
        if stride not in (1, 2):
            raise ValueError("stride must be 1 or 2")
        self.stride = stride
        hidden_dim = round(inp * expansion)
        self.use_res_connect = self.stride == 1 and inp == oup
        if expansion == 1:
            self.conv = nn.Sequential(
                nn.Conv2d(hidden_dim, hidden_dim, 3, stride, 1, groups=hidden_dim, dilation=dilation, bias=False),
                nn.BatchNorm2d(hidden_dim),
                nn.ReLU6(inplace=True),
                nn.Conv2d(hidden_dim, oup, 1, 1, 0, bias=False),
                nn.BatchNorm2d(oup),
            )
        else:
            self.conv = nn.Sequential(
                nn.Conv2d(inp, hidden_dim, 1, 1, 0, bias=False),
                nn.BatchNorm2d(hidden_dim),
                nn.ReLU6(inplace=True),
                nn.Conv2d(hidden_dim, hidden_dim, 3, stride, 1, groups=hidden_dim, dilation=dilation, bias=False),
                nn.BatchNorm2d(hidden_dim),
                nn.ReLU6(inplace=True),
                nn.Conv2d(hidden_dim, oup, 1, 1, 0, bias=False),
                nn.BatchNorm2d(oup),
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.conv(x) if self.use_res_connect else self.conv(x)


class MobileNetV2(nn.Module):
    """The feature extractor only (`num_classes=None` upstream): 19 stages under `features`."""

    def __init__(self, in_channels: int, alpha: float = 1.0, expansion: int = 6) -> None:
        super().__init__()
        self.in_channels = in_channels
        input_channel = _make_divisible(32 * alpha, 8)
        self.last_channel = _make_divisible(1280 * alpha, 8) if alpha > 1.0 else 1280
        settings = [  # t, c, n, s
            [1, 16, 1, 1],
            [expansion, 24, 2, 2],
            [expansion, 32, 3, 2],
            [expansion, 64, 4, 2],
            [expansion, 96, 3, 1],
            [expansion, 160, 3, 2],
            [expansion, 320, 1, 1],
        ]
        features: list[nn.Module] = [_conv_bn(self.in_channels, input_channel, 2)]
        for t, c, n, s in settings:
            output_channel = _make_divisible(int(c * alpha), 8)
            for i in range(n):
                features.append(InvertedResidual(input_channel, output_channel, s if i == 0 else 1, expansion=t))
                input_channel = output_channel
        features.append(_conv_1x1_bn(input_channel, self.last_channel))
        self.features = nn.Sequential(*features)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.features(x)


class MobileNetV2Backbone(nn.Module):
    """Upstream `MobileNetV2Backbone`: returns the five encoder scales (2×, 4×, 8×, 16×, 32×)."""

    def __init__(self, in_channels: int) -> None:
        super().__init__()
        self.in_channels = in_channels
        self.model = MobileNetV2(self.in_channels, alpha=1.0, expansion=6)
        self.enc_channels = [16, 24, 32, 96, 1280]

    def forward(self, x: torch.Tensor) -> list[torch.Tensor]:
        features = self.model.features
        out = []
        for start, stop in ((0, 2), (2, 4), (4, 7), (7, 14), (14, 19)):
            for index in range(start, stop):
                x = features[index](x)
            out.append(x)
        return out


# --------------------------------------------------------------------------------------------------
# MODNet basic modules
# --------------------------------------------------------------------------------------------------


class IBNorm(nn.Module):
    """Instance norm and batch norm side by side over two halves of the channels."""

    def __init__(self, in_channels: int) -> None:
        super().__init__()
        self.bnorm_channels = int(in_channels / 2)
        self.inorm_channels = in_channels - self.bnorm_channels
        self.bnorm = nn.BatchNorm2d(self.bnorm_channels, affine=True)
        self.inorm = nn.InstanceNorm2d(self.inorm_channels, affine=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        bn_x = self.bnorm(x[:, : self.bnorm_channels, ...].contiguous())
        in_x = self.inorm(x[:, self.bnorm_channels :, ...].contiguous())
        return torch.cat((bn_x, in_x), 1)


class Conv2dIBNormRelu(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int,
        stride: int = 1,
        padding: int = 0,
        dilation: int = 1,
        groups: int = 1,
        bias: bool = True,
        with_ibn: bool = True,
        with_relu: bool = True,
    ) -> None:
        super().__init__()
        layers: list[nn.Module] = [
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size,
                stride=stride,
                padding=padding,
                dilation=dilation,
                groups=groups,
                bias=bias,
            )
        ]
        if with_ibn:
            layers.append(IBNorm(out_channels))
        if with_relu:
            layers.append(nn.ReLU(inplace=True))
        self.layers = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.layers(x)


class SEBlock(nn.Module):
    """Squeeze-and-excitation (Hu et al., 2018)."""

    def __init__(self, in_channels: int, out_channels: int, reduction: int = 1) -> None:
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(in_channels, int(in_channels // reduction), bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(int(in_channels // reduction), out_channels, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, _, _ = x.size()
        w = self.pool(x).view(b, c)
        w = self.fc(w).view(b, c, 1, 1)
        return x * w.expand_as(x)


# --------------------------------------------------------------------------------------------------
# MODNet branches
# --------------------------------------------------------------------------------------------------


class LRBranch(nn.Module):
    """Low-resolution (semantic) branch."""

    def __init__(self, backbone: MobileNetV2Backbone) -> None:
        super().__init__()
        enc_channels = backbone.enc_channels
        self.backbone = backbone
        self.se_block = SEBlock(enc_channels[4], enc_channels[4], reduction=4)
        self.conv_lr16x = Conv2dIBNormRelu(enc_channels[4], enc_channels[3], 5, stride=1, padding=2)
        self.conv_lr8x = Conv2dIBNormRelu(enc_channels[3], enc_channels[2], 5, stride=1, padding=2)
        self.conv_lr = Conv2dIBNormRelu(enc_channels[2], 1, kernel_size=3, stride=2, padding=1, with_ibn=False, with_relu=False)

    def forward(self, img: torch.Tensor, inference: bool):
        enc_features = self.backbone(img)
        enc2x, enc4x, enc32x = enc_features[0], enc_features[1], enc_features[4]
        enc32x = self.se_block(enc32x)
        lr16x = F.interpolate(enc32x, scale_factor=2, mode="bilinear", align_corners=False)
        lr16x = self.conv_lr16x(lr16x)
        lr8x = F.interpolate(lr16x, scale_factor=2, mode="bilinear", align_corners=False)
        lr8x = self.conv_lr8x(lr8x)
        pred_semantic = None
        if not inference:
            pred_semantic = torch.sigmoid(self.conv_lr(lr8x))
        return pred_semantic, lr8x, [enc2x, enc4x]


class HRBranch(nn.Module):
    """High-resolution (detail) branch."""

    def __init__(self, hr_channels: int, enc_channels: list[int]) -> None:
        super().__init__()
        self.tohr_enc2x = Conv2dIBNormRelu(enc_channels[0], hr_channels, 1, stride=1, padding=0)
        self.conv_enc2x = Conv2dIBNormRelu(hr_channels + 3, hr_channels, 3, stride=2, padding=1)
        self.tohr_enc4x = Conv2dIBNormRelu(enc_channels[1], hr_channels, 1, stride=1, padding=0)
        self.conv_enc4x = Conv2dIBNormRelu(2 * hr_channels, 2 * hr_channels, 3, stride=1, padding=1)
        self.conv_hr4x = nn.Sequential(
            Conv2dIBNormRelu(3 * hr_channels + 3, 2 * hr_channels, 3, stride=1, padding=1),
            Conv2dIBNormRelu(2 * hr_channels, 2 * hr_channels, 3, stride=1, padding=1),
            Conv2dIBNormRelu(2 * hr_channels, hr_channels, 3, stride=1, padding=1),
        )
        self.conv_hr2x = nn.Sequential(
            Conv2dIBNormRelu(2 * hr_channels, 2 * hr_channels, 3, stride=1, padding=1),
            Conv2dIBNormRelu(2 * hr_channels, hr_channels, 3, stride=1, padding=1),
            Conv2dIBNormRelu(hr_channels, hr_channels, 3, stride=1, padding=1),
            Conv2dIBNormRelu(hr_channels, hr_channels, 3, stride=1, padding=1),
        )
        self.conv_hr = nn.Sequential(
            Conv2dIBNormRelu(hr_channels + 3, hr_channels, 3, stride=1, padding=1),
            Conv2dIBNormRelu(hr_channels, 1, kernel_size=1, stride=1, padding=0, with_ibn=False, with_relu=False),
        )

    def forward(self, img: torch.Tensor, enc2x: torch.Tensor, enc4x: torch.Tensor, lr8x: torch.Tensor, inference: bool):
        img2x = F.interpolate(img, scale_factor=1 / 2, mode="bilinear", align_corners=False)
        img4x = F.interpolate(img, scale_factor=1 / 4, mode="bilinear", align_corners=False)
        enc2x = self.tohr_enc2x(enc2x)
        hr4x = self.conv_enc2x(torch.cat((img2x, enc2x), dim=1))
        enc4x = self.tohr_enc4x(enc4x)
        hr4x = self.conv_enc4x(torch.cat((hr4x, enc4x), dim=1))
        lr4x = F.interpolate(lr8x, scale_factor=2, mode="bilinear", align_corners=False)
        hr4x = self.conv_hr4x(torch.cat((hr4x, lr4x, img4x), dim=1))
        hr2x = F.interpolate(hr4x, scale_factor=2, mode="bilinear", align_corners=False)
        hr2x = self.conv_hr2x(torch.cat((hr2x, enc2x), dim=1))
        pred_detail = None
        if not inference:
            hr = F.interpolate(hr2x, scale_factor=2, mode="bilinear", align_corners=False)
            hr = self.conv_hr(torch.cat((hr, img), dim=1))
            pred_detail = torch.sigmoid(hr)
        return pred_detail, hr2x


class FusionBranch(nn.Module):
    """Semantic-detail fusion branch: the alpha matte."""

    def __init__(self, hr_channels: int, enc_channels: list[int]) -> None:
        super().__init__()
        self.conv_lr4x = Conv2dIBNormRelu(enc_channels[2], hr_channels, 5, stride=1, padding=2)
        self.conv_f2x = Conv2dIBNormRelu(2 * hr_channels, hr_channels, 3, stride=1, padding=1)
        self.conv_f = nn.Sequential(
            Conv2dIBNormRelu(hr_channels + 3, int(hr_channels / 2), 3, stride=1, padding=1),
            Conv2dIBNormRelu(int(hr_channels / 2), 1, 1, stride=1, padding=0, with_ibn=False, with_relu=False),
        )

    def forward(self, img: torch.Tensor, lr8x: torch.Tensor, hr2x: torch.Tensor) -> torch.Tensor:
        lr4x = F.interpolate(lr8x, scale_factor=2, mode="bilinear", align_corners=False)
        lr4x = self.conv_lr4x(lr4x)
        lr2x = F.interpolate(lr4x, scale_factor=2, mode="bilinear", align_corners=False)
        f2x = self.conv_f2x(torch.cat((lr2x, hr2x), dim=1))
        f = F.interpolate(f2x, scale_factor=2, mode="bilinear", align_corners=False)
        f = self.conv_f(torch.cat((f, img), dim=1))
        return torch.sigmoid(f)


class MODNet(nn.Module):
    """MODNet: a MobileNetV2 low-resolution branch, a high-resolution detail branch and a fusion branch.

    `forward(img, inference)` returns `(pred_semantic, pred_detail, pred_matte)`; with `inference=True` the first
    two are `None` (the heads that produce them are skipped, as upstream). `img` is (B, 3, H, W) normalised to
    [-1, 1] with H and W multiples of 32.
    """

    def __init__(self, in_channels: int = 3, hr_channels: int = 32) -> None:
        super().__init__()
        self.in_channels = in_channels
        self.hr_channels = hr_channels
        self.backbone = MobileNetV2Backbone(self.in_channels)
        self.lr_branch = LRBranch(self.backbone)
        self.hr_branch = HRBranch(self.hr_channels, self.backbone.enc_channels)
        self.f_branch = FusionBranch(self.hr_channels, self.backbone.enc_channels)

    def forward(self, img: torch.Tensor, inference: bool = True):
        pred_semantic, lr8x, (enc2x, enc4x) = self.lr_branch(img, inference)
        pred_detail, hr2x = self.hr_branch(img, enc2x, enc4x, lr8x, inference)
        pred_matte = self.f_branch(img, lr8x, hr2x)
        return pred_semantic, pred_detail, pred_matte

    def freeze_norm(self) -> None:
        """Put every BatchNorm / InstanceNorm layer in eval mode (upstream `freeze_norm`)."""
        for module in self.modules():
            if isinstance(module, nn.BatchNorm2d | nn.InstanceNorm2d):
                module.eval()

**Module 3/4:** `src/modnet_matting_pipeline/pipeline.py` (carried verbatim; see the note above)

In [ ]:
"""MODNet photographic portrait matting (`ZHKKKe/MODNet`, `modnet_photographic_portrait_matting.ckpt`) DIMER
pipeline: verified snapshot, static audit and one-time conversion of the legacy torch pickle into safetensors,
trimap-free alpha-matte prediction, held-out evaluation against constant baselines, and bounded fine-tuning of the
matting branches to labelled portrait/alpha pairs with a portable adapter.

MODNet (Ke et al., AAAI 2022) predicts a portrait's alpha matte from an RGB image alone — no trimap — with a
MobileNetV2 low-resolution semantic branch, a high-resolution detail branch and a fusion branch. The authors
publish the photographic checkpoint on Google Drive; the byte-identical file is mirrored on the Hugging Face Hub
(`XM5354/Modnet_models`), which is what this package pins at an immutable revision with the digest of the Drive
original recorded beside it (docs/WEIGHTS.md).

The asset is a **legacy torch.save file** (not a zip archive): five pickle streams — the magic number, the protocol
version, the system record, the `OrderedDict` state dict whose tensors are persistent-id references to
`torch.FloatStorage` / `torch.LongStorage`, and the storage keys — followed by raw storage bytes. Under the fleet
asset specification (§11) that is executable serialization, so this package converts it once — every header
stream statically audited against an allow-list, then `torch.load(weights_only=True)`, the DataParallel `module.`
prefix stripped, a strict load into the vendored architecture — into safetensors with a pinned digest, and serves
only the converted file. The architecture is vendored in `modeling.py` from the upstream repository at a pinned
commit; nothing is fetched from the Hub at load time except the manifest-listed file.

Everything model-related is imported lazily so that snapshot verification, the pickle audit and input validation
run (and can refuse) before `torch` is imported (fleet RTM-001). `numpy` and Pillow are used for images and are
imported freely.
"""

from __future__ import annotations

import hashlib
import io
import json
import math
import pickletools
import time
import zipfile
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

MODEL_ID = "XM5354/Modnet_models"
MODEL_REVISION = "71aca6d04ed0267b4b12bde776868f2b9fb1d06f"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "modnet-photographic-portrait-matting"
ARTIFACT_FORMAT = "org.valcorza.modnet-matting.adapter.v1"
ARTIFACT_FORMAT_VERSION = "1.0"
ARTIFACT_WEIGHTS_NAME = "adapter.safetensors"
ARTIFACT_MANIFEST_NAME = "manifest.json"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# Immutable upstream source asset (the authors' Google Drive file; the Hub mirror is byte-identical, docs/WEIGHTS.md).
SOURCE_CKPT_NAME = "modnet_photographic_portrait_matting.ckpt"
SOURCE_CKPT_BYTES = 26_255_603
SOURCE_CKPT_SHA256 = "7c22235f0925deba15d4d63e53afcb654c47055bbcd98f56e393ab2584007ed8"
SOURCE_DRIVE_FILE_ID = "1mcr7ALciuAsHCpLnrtG_eop5-EYhbCmz"  # in the authors' folder 1umYmlCulvIFNaqPjwod1SayFmSRHziyR
UPSTREAM_CODE_COMMIT = "28165a451e4610c9d77cfdf925a94610bb2810fb"  # ZHKKKe/MODNet, the vendored architecture
# Code-free serving file produced deterministically by `convert_model` (asset spec §11.2).
CONVERTED_WEIGHTS_NAME = "modnet-photographic-portrait-matting.safetensors"
CONVERTED_SHA256 = "0ec6d832a873ea38974077b23920da806cc2e44e5553c7ee12e7cb7b921bcc6b"
CONVERTED_BYTES = 26_135_396
# Static-audit digest of the source pickle (sorted global names), see `audit_pickle`.
PICKLE_AUDIT_SHA256 = "5b9f0ba08490293d6c17b9cef219991e1a6edda31609429679f8dca1af5a7b10"
CKPT_ALLOWED_GLOBALS = frozenset(
    {"collections.OrderedDict", "torch._utils._rebuild_tensor_v2", "torch.FloatStorage", "torch.LongStorage"}
)
LEGACY_MAGIC_NUMBER = 0x1950A86A20F9469CFC6C  # torch's legacy serialization header
LEGACY_HEADER_STREAMS = 5  # magic, protocol version, sys_info, the object, the storage keys
STATE_DICT_PREFIX = "module."  # the checkpoint was saved from an nn.DataParallel wrapper
ALIAS_PREFIX = "lr_branch.backbone."  # the LR branch holds the same backbone module; these keys alias `backbone.`

# Architecture and data-contract facts.
HR_CHANNELS = 32
PARAMETER_COUNT = 6_487_075  # nn.Parameters of the vendored MODNet (shared backbone counted once)
STATE_TENSORS = 439  # tensors in the converted file (the state dict without the 312 aliased LR-branch backbone keys)
STATE_NUMEL = 6_521_976  # elements in the converted file (parameters + BatchNorm buffers)
CHECKPOINT_TENSORS = 751  # tensors in the source state dict (aliases included)
REF_SIZE = 512  # upstream inference resizes the short side to 512 (and both sides to multiples of 32)
TRAIN_SIZE = 512  # labelled records are exactly this square size
MIN_SIDE, MAX_SIDE = 64, 4_096  # inference images
MIN_RECORDS = 4
MAX_RECORDS = 2_000
ADAPTATION_MODES = ("branches", "full")  # the only scopes an adapter may declare
FROZEN_PREFIXES: dict[str, tuple[str, ...]] = {"branches": ("backbone.", ALIAS_PREFIX), "full": ()}
TRIMAP_RADIUS = 8  # pixels of unknown band grown around the alpha transition (at TRAIN_SIZE)
# Upstream's `GaussianBlurLayer(1, 3)` kernel: scipy.ndimage.gaussian_filter of a 3 × 3 delta with sigma 0.8
# (0.3 · ((3 − 1) · 0.5 − 1) + 0.8), reflect mode — hard-coded so scipy is not a dependency.
SEMANTIC_BLUR_KERNEL: tuple[tuple[float, ...], ...] = (
    (0.06261056416945447, 0.12499990229092141, 0.06261056416945447),
    (0.12499990229092141, 0.24955813415849687, 0.12499990229092141),
    (0.06261056416945447, 0.12499990229092141, 0.06261056416945447),
)
LOSS_SCALES = {"semantic": 10.0, "detail": 10.0, "matte": 1.0}  # upstream `supervised_training_iter` defaults


# --------------------------------------------------------------------------------------------------
# manifest, staging, static pickle audit and conversion
# --------------------------------------------------------------------------------------------------


def _sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _verify_manifest(root: Path, model_id: str, revision: str) -> dict[str, Any]:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"no snapshot manifest at {manifest_path}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("modelId") != model_id:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {model_id!r}")
    if manifest.get("revision") != revision:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {revision!r}")
    listed = {entry["path"] for entry in manifest["files"]}
    if SOURCE_CKPT_NAME not in listed:
        raise ValueError(f"manifest does not list {SOURCE_CKPT_NAME}; refusing to proceed")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256_file(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
        if entry["path"] == SOURCE_CKPT_NAME and (size, digest) != (SOURCE_CKPT_BYTES, SOURCE_CKPT_SHA256):
            raise ValueError(f"{entry['path']}: manifest digest disagrees with the package constant")
    return manifest


def verify_converted(path: str | Path | None = None) -> dict[str, Any]:
    """Check the converted serving file (safetensors) against the pinned digest."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    file_path = root / CONVERTED_WEIGHTS_NAME
    if not file_path.is_file():
        raise FileNotFoundError(f"converted file missing: {file_path}")
    size = file_path.stat().st_size
    if size != CONVERTED_BYTES:
        raise ValueError(f"{CONVERTED_WEIGHTS_NAME}: size {size} != pinned {CONVERTED_BYTES}")
    digest = _sha256_file(file_path)
    if digest != CONVERTED_SHA256:
        raise ValueError(f"{CONVERTED_WEIGHTS_NAME}: sha256 {digest} != pinned {CONVERTED_SHA256}")
    return {"files": [{"path": CONVERTED_WEIGHTS_NAME, "bytes": size, "sha256": digest}]}


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check the snapshot against its DIMER manifest (size + SHA-256 of every listed Hub file) and, when the
    converted serving file is present, that against the pinned digest."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _verify_manifest(root, MODEL_ID, MODEL_REVISION)
    converted = (root / CONVERTED_WEIGHTS_NAME).is_file()
    if converted:
        verify_converted(root)
    return {**manifest, "converted": converted}


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at the pinned revision straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest entries that are absent locally (a fresh clone commits the manifest and git-ignores the
    25 MB checkpoint and the safetensors it converts to)."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def _pickle_stream_globals(stream: io.BytesIO) -> tuple[dict[str, int], Any]:
    """Globals of one pickle stream read from the current position (stops at STOP, leaving the stream after it),
    plus the value of a single LONG1/LONG/INT top-level constant (for the legacy magic number) when the stream is
    that simple. Collected with `pickletools.genops` — no execution."""
    found: dict[str, int] = {}
    stack: list[Any] = []
    constant = None
    n_ops = 0
    for op, arg, _pos in pickletools.genops(stream):
        n_ops += 1
        if op.name == "GLOBAL":  # pickletools renders the (module, name) pair space-separated
            key = arg.replace("\n", " ").replace(" ", ".", 1)
            found[key] = found.get(key, 0) + 1
        elif op.name == "STACK_GLOBAL":
            key = f"{stack[-2]}.{stack[-1]}"
            found[key] = found.get(key, 0) + 1
        if op.name in ("SHORT_BINUNICODE", "BINUNICODE", "UNICODE", "SHORT_BINSTRING", "BINSTRING"):
            stack.append(arg)
        elif op.name in ("LONG1", "LONG4", "LONG", "INT", "BININT", "BININT1", "BININT2"):
            constant = arg
            stack.append(None)
        elif op.name in ("MEMOIZE", "BINPUT", "LONG_BINPUT", "PUT"):
            pass
        else:
            stack.append(None)
        if op.name == "STOP":
            break
    return found, (constant if n_ops <= 3 else None)


def _pickle_globals(data: bytes) -> dict[str, int]:
    found, _ = _pickle_stream_globals(io.BytesIO(data))
    return found


def _legacy_globals(data: bytes) -> tuple[dict[str, int], int]:
    """Globals of the five header streams of a legacy torch.save file (the raw storage bytes after them are not
    pickles and are not read)."""
    stream = io.BytesIO(data)
    found: dict[str, int] = {}
    for index in range(LEGACY_HEADER_STREAMS):
        part, constant = _pickle_stream_globals(stream)
        if index == 0 and constant != LEGACY_MAGIC_NUMBER:
            raise ValueError("not a legacy torch.save file: the first pickle stream is not the magic number")
        for key, count in part.items():
            found[key] = found.get(key, 0) + count
    return found, stream.tell()


def audit_pickle(path: str | Path, *, allowed: frozenset[str] = CKPT_ALLOWED_GLOBALS) -> dict[str, Any]:
    """Statically list the globals a pickle (plain, inside a torch zip archive, or the header streams of a legacy
    torch.save file) would import and refuse any outside `allowed`. Executes nothing. Returns the sorted globals
    and their digest."""
    file_path = Path(path)
    if not file_path.is_file():
        raise FileNotFoundError(f"file not found: {file_path}")
    data = file_path.read_bytes()
    found: dict[str, int] = {}
    layout = "pickle"
    pickles = 1
    header_bytes = len(data)
    if data[:4] == b"PK\x03\x04":
        layout = "torch_zip"
        pickles = 0
        archive = zipfile.ZipFile(io.BytesIO(data))
        for name in archive.namelist():
            if name.endswith(".pkl"):
                pickles += 1
                for key, count in _pickle_globals(archive.read(name)).items():
                    found[key] = found.get(key, 0) + count
    else:
        _first, constant = _pickle_stream_globals(io.BytesIO(data))
        if constant == LEGACY_MAGIC_NUMBER:
            layout = "torch_legacy"
            pickles = LEGACY_HEADER_STREAMS
            found, header_bytes = _legacy_globals(data)
        else:
            found = _pickle_globals(data)
    violations = sorted(name for name in found if name not in allowed)
    summary = {
        "file": file_path.name,
        "layout": layout,
        "pickles": pickles,
        "header_bytes": header_bytes,
        "globals": sorted(found),
        "violations": violations,
        "audit_sha256": hashlib.sha256("\n".join(sorted(found)).encode("utf-8")).hexdigest(),
    }
    if violations:
        raise ValueError(f"{file_path.name}: pickle audit failed, globals outside the allow-list: {violations}")
    return summary


def _check_pinned_source(root: Path) -> dict[str, Any]:
    source = root / SOURCE_CKPT_NAME
    if not source.is_file():
        raise FileNotFoundError(f"source file not found: {source}")
    size = source.stat().st_size
    if size != SOURCE_CKPT_BYTES:
        raise ValueError(f"{SOURCE_CKPT_NAME}: size {size} != pinned {SOURCE_CKPT_BYTES}")
    digest = _sha256_file(source)
    if digest != SOURCE_CKPT_SHA256:
        raise ValueError(f"{SOURCE_CKPT_NAME}: sha256 {digest} != pinned {SOURCE_CKPT_SHA256}")
    audit = audit_pickle(source)
    if audit["audit_sha256"] != PICKLE_AUDIT_SHA256:
        raise ValueError(f"{SOURCE_CKPT_NAME}: pickle audit digest {audit['audit_sha256']} != pinned {PICKLE_AUDIT_SHA256}")
    return {"path": SOURCE_CKPT_NAME, "bytes": size, "sha256": digest, "audit": audit}


def build_model() -> Any:
    """Instantiate the vendored MODNet architecture (random initialisation; no download, no pickle)."""
    pass  # standalone rewrite (build_notebook.py): `from .modeling import MODNet` removed — names are kernel globals defined by the carried modules

    return MODNet(in_channels=3, hr_channels=HR_CHANNELS)


def _expand_aliases(state: Mapping[str, Any]) -> dict[str, Any]:
    """The model's state dict lists the shared backbone twice (`backbone.*` and `lr_branch.backbone.*`); the
    converted file stores it once and the aliases are re-created here."""
    out = dict(state)
    for key, value in state.items():
        if key.startswith("backbone."):
            out[ALIAS_PREFIX + key[len("backbone.") :]] = value
    return out


def convert_model(path: str | Path | None = None) -> dict[str, Any]:
    """Convert the pinned legacy checkpoint into safetensors, deterministically, after size, digest and
    static-audit checks: torch's weights-only unpickler, the `module.` prefix stripped, aliased backbone keys
    checked equal and dropped, a strict load into the vendored architecture, and the model's own state dict
    (backbone once) saved."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    source = _check_pinned_source(root)
    import torch
    from safetensors.torch import save_file

    started = time.perf_counter()
    payload = torch.load(root / SOURCE_CKPT_NAME, map_location="cpu", weights_only=True)
    if not isinstance(payload, dict) or any(not isinstance(v, torch.Tensor) for v in payload.values()):
        raise ValueError(f"{SOURCE_CKPT_NAME} did not unpickle to a state dict of tensors")
    if len(payload) != CHECKPOINT_TENSORS:
        raise ValueError(f"{SOURCE_CKPT_NAME}: {len(payload)} tensors, expected {CHECKPOINT_TENSORS}")
    stripped = {}
    for key, value in payload.items():
        if not key.startswith(STATE_DICT_PREFIX):
            raise ValueError(f"{SOURCE_CKPT_NAME}: unexpected state-dict key {key!r} outside {STATE_DICT_PREFIX!r}")
        stripped[key[len(STATE_DICT_PREFIX) :]] = value
    for key, value in stripped.items():
        if key.startswith(ALIAS_PREFIX):
            twin = "backbone." + key[len(ALIAS_PREFIX) :]
            if twin not in stripped or not torch.equal(stripped[twin], value):
                raise ValueError(f"{SOURCE_CKPT_NAME}: aliased key {key!r} differs from {twin!r}")
    model = build_model()
    model.load_state_dict(stripped, strict=True)
    canonical = {k: v.contiguous() for k, v in model.state_dict().items() if not k.startswith(ALIAS_PREFIX)}
    n_elements = sum(v.numel() for v in canonical.values())
    if len(canonical) != STATE_TENSORS or n_elements != STATE_NUMEL:
        raise ValueError(
            f"converted state dict has {len(canonical)} tensors / {n_elements} elements; expected {STATE_TENSORS} / {STATE_NUMEL}"
        )
    save_file(canonical, str(root / CONVERTED_WEIGHTS_NAME), metadata={"format": "pt"})
    report = verify_converted(root)
    return {
        "source": {k: v for k, v in source.items() if k != "audit"},
        "audit": source["audit"],
        "checkpoint": {
            "tensors": len(payload),
            "prefix": STATE_DICT_PREFIX,
            "aliased_backbone_tensors": len(payload) - STATE_TENSORS,
        },
        "converted": report["files"],
        "seconds": round(time.perf_counter() - started, 2),
    }


# --------------------------------------------------------------------------------------------------
# images, mattes and validation (no model import)
# --------------------------------------------------------------------------------------------------

INPUT_SCHEMA: dict[str, Any] = {
    "record": (
        "{id, image, alpha?}: image = (H, W, 3) uint8 RGB array or an image file path (JPEG/PNG; greyscale and RGBA are "
        "converted to RGB); alpha = (H, W) float in [0, 1] (or an 8-bit greyscale PNG path, scaled by 1/255), optional"
    ),
    "inference_size": (
        f"any image with both sides in [{MIN_SIDE}, {MAX_SIDE}]; resized as upstream (short side {REF_SIZE}, multiples of 32) "
        "and the matte returned at the input size"
    ),
    "training_size": f"labelled records must be exactly {TRAIN_SIZE} × {TRAIN_SIZE} (the BYOD loader resizes pairs)",
    "records": [MIN_RECORDS, MAX_RECORDS],
    "validation": (
        "record shape, dtype, size range, finiteness and alpha range only. Nothing checks that the image shows a "
        "portrait, that the alpha was drawn for that image, or that the alpha marks the person rather than something "
        "else -- any RGB image is matted without complaint"
    ),
}


def read_image(path: str | Path) -> Any:
    """Load an image file with Pillow as (H, W, 3) uint8 RGB."""
    import numpy as np
    from PIL import Image

    with Image.open(path) as im:
        return np.asarray(im.convert("RGB"), dtype=np.uint8)


def read_alpha(path: str | Path) -> Any:
    """Load an 8-bit greyscale alpha PNG as (H, W) float32 in [0, 1]."""
    import numpy as np
    from PIL import Image

    with Image.open(path) as im:
        return np.asarray(im.convert("L"), dtype=np.float32) / 255.0


def _check_record(record: Any, index: int, *, training: bool) -> dict[str, Any]:
    import numpy as np

    label_name = f"records[{index}]"
    if not isinstance(record, Mapping):
        raise ValueError(f"{label_name} must be a mapping with id/image[/alpha]")
    for key in ("id", "image"):
        if key not in record:
            raise ValueError(f"{label_name} is missing {key!r}")
    rid, image = record["id"], record["image"]
    if not isinstance(rid, str) or not rid or len(rid) > 128:
        raise ValueError(f"{label_name}: id must be a non-empty string of at most 128 characters")
    if isinstance(image, str | Path):
        if not Path(image).is_file():
            raise ValueError(f"{label_name}: image file not found: {image}")
        image = read_image(image)
    try:
        array = np.asarray(image)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"{label_name}: image must be an array") from exc
    if array.ndim != 3 or array.shape[2] != 3:
        raise ValueError(f"{label_name}: image must have shape (H, W, 3), got {array.shape}")
    if array.dtype != np.uint8:
        raise ValueError(f"{label_name}: image must be uint8 RGB, got dtype {array.dtype}")
    h, w = array.shape[:2]
    if training:
        if (h, w) != (TRAIN_SIZE, TRAIN_SIZE):
            raise ValueError(f"{label_name}: a labelled record must be {TRAIN_SIZE} × {TRAIN_SIZE}, got {h} × {w}")
    elif not (MIN_SIDE <= h <= MAX_SIDE and MIN_SIDE <= w <= MAX_SIDE):
        raise ValueError(f"{label_name}: image sides must be in [{MIN_SIDE}, {MAX_SIDE}], got {h} × {w}")
    item: dict[str, Any] = {"id": rid, "image": np.ascontiguousarray(array)}
    alpha = record.get("alpha")
    if alpha is not None:
        if isinstance(alpha, str | Path):
            if not Path(alpha).is_file():
                raise ValueError(f"{label_name}: alpha file not found: {alpha}")
            alpha = read_alpha(alpha)
        try:
            matte = np.asarray(alpha, dtype=np.float32)
        except (TypeError, ValueError) as exc:
            raise ValueError(f"{label_name}: alpha must be a numeric array") from exc
        if matte.shape != (h, w):
            raise ValueError(f"{label_name}: alpha must have shape {(h, w)}, got {matte.shape}")
        if not np.all(np.isfinite(matte)):
            raise ValueError(f"{label_name}: alpha contains non-finite values")
        if float(matte.min()) < 0.0 or float(matte.max()) > 1.0:
            raise ValueError(
                f"{label_name}: alpha values must lie in [0, 1], got [{float(matte.min()):.3f}, {float(matte.max()):.3f}]"
            )
        item["alpha"] = np.ascontiguousarray(matte)
    elif training:
        raise ValueError(f"{label_name} has no alpha; every record of a labelled dataset needs one")
    for key in ("title", "source", "license", "meta", "original_size", "loaded_size"):
        if key in record:
            item[key] = record[key]
    return item


def check_record(record: Mapping[str, Any], *, training: bool = False) -> dict[str, Any]:
    """Validate one record and return its normalised copy (uint8 RGB image, float32 alpha)."""
    return _check_record(record, 0, training=training)


def record_digest(record: Mapping[str, Any], *, training: bool = False) -> str:
    checked = _check_record(record, 0, training=training)
    digest = hashlib.sha256(checked["image"].tobytes())
    if "alpha" in checked:
        digest.update(checked["alpha"].tobytes())
    return digest.hexdigest()


def dataset_digest(records: Sequence[Mapping[str, Any]], *, training: bool = True) -> str:
    payload = [[r["id"], record_digest(r, training=training)] for r in records]
    return hashlib.sha256(json.dumps(payload, separators=(",", ":")).encode("utf-8")).hexdigest()


def validate_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    min_records: int = MIN_RECORDS,
    max_records: int = MAX_RECORDS,
    require_labels: bool = True,
) -> dict[str, Any]:
    """Structural validation of a portrait dataset; raises ValueError before any model import. Labelled datasets
    (`require_labels=True`) must be `TRAIN_SIZE` squares with an alpha per record."""

    if isinstance(records, Mapping) or not isinstance(records, Sequence) or isinstance(records, str | bytes):
        raise ValueError("records must be a list of {id, image, alpha} mappings")
    if not min_records <= len(records) <= max_records:
        raise ValueError(f"{len(records)} records; {min_records}..{max_records} are required")
    checked = []
    ids: set[str] = set()
    alpha_sum = 0.0
    fractional = 0
    pixels = 0
    for index, record in enumerate(records):
        item = _check_record(record, index, training=require_labels)
        if item["id"] in ids:
            raise ValueError(f"duplicate id {item['id']!r}")
        ids.add(item["id"])
        if "alpha" in item:
            alpha_sum += float(item["alpha"].sum())
            fractional += int(((item["alpha"] > 0.02) & (item["alpha"] < 0.98)).sum())
            pixels += item["alpha"].size
        checked.append(item)
    labelled = sum("alpha" in r for r in checked)
    if require_labels and alpha_sum == 0.0:
        raise ValueError("every alpha is all background; nothing to learn or evaluate")
    sizes = sorted({(int(r["image"].shape[0]), int(r["image"].shape[1])) for r in checked})
    return {
        "records": checked,
        "n_records": len(checked),
        "n_labelled": labelled,
        "sizes": [list(s) for s in sizes],
        "foreground_fraction": round(alpha_sum / pixels, 4) if pixels else None,
        "fractional_alpha_fraction": round(fractional / pixels, 4) if pixels else None,
        "digest": dataset_digest(checked, training=require_labels),
        "model_id": MODEL_ID,
    }


def validate_inputs(record: Mapping[str, Any]) -> dict[str, Any]:
    """Validate one record (inference contract); returns its id, size and, with an alpha, the foreground fraction."""
    item = _check_record(record, 0, training=False)
    report: dict[str, Any] = {"id": item["id"], "shape": tuple(int(x) for x in item["image"].shape), "has_alpha": "alpha" in item}
    if "alpha" in item:
        report["foreground_fraction"] = round(float(item["alpha"].mean()), 4)
    return report


def trimap_from_alpha(alpha: Any, *, radius: int = TRIMAP_RADIUS) -> Any:
    """Upstream-style trimap from a ground-truth matte: 1 = foreground, 0 = background, 0.5 = unknown, the unknown
    band being every pixel whose alpha is fractional grown by `radius` pixels (a max filter) — numpy only."""
    import numpy as np

    a = np.asarray(alpha, dtype=np.float32)
    unknown = (a > 0.02) & (a < 0.98)
    if radius > 0:
        padded = np.pad(unknown, radius, mode="constant", constant_values=False)
        grown = np.zeros_like(unknown)
        for dy in range(-radius, radius + 1):
            for dx in range(-radius, radius + 1):
                if dy * dy + dx * dx <= radius * radius:
                    grown |= padded[radius + dy : radius + dy + a.shape[0], radius + dx : radius + dx + a.shape[1]]
        unknown = grown
    trimap = np.where(a >= 0.5, 1.0, 0.0).astype(np.float32)
    trimap[unknown] = 0.5
    return trimap


# --------------------------------------------------------------------------------------------------
# pipeline
# --------------------------------------------------------------------------------------------------


def _inference_size(h: int, w: int) -> tuple[int, int]:
    """Upstream `inference.py`: short side to REF_SIZE unless the image already straddles it; multiples of 32."""
    if max(h, w) < REF_SIZE or min(h, w) > REF_SIZE:
        if w >= h:
            rh, rw = REF_SIZE, int(w / h * REF_SIZE)
        else:
            rw, rh = REF_SIZE, int(h / w * REF_SIZE)
    else:
        rh, rw = h, w
    return max(32, rh - rh % 32), max(32, rw - rw % 32)


@dataclass
class ModNetMattingPipeline:
    """Trimap-free portrait matting and bounded fine-tuning on top of the verified MODNet checkpoint."""

    model: Any
    device: str
    weights_dir: Path
    source: str
    adapter: dict[str, Any] | None = None

    @classmethod
    def from_pretrained(
        cls,
        *,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
        require_source: bool = True,
        report: Callable[[dict[str, Any]], None] | None = None,
    ) -> ModNetMattingPipeline:
        """Verify, convert if needed, build the vendored architecture and strictly load. With `require_source=False`
        the checkpoint may be absent (the DIMER-hosted case) as long as the converted file verifies. `report`
        receives the audit and conversion records when a conversion happens."""
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if require_source:
            stage_missing_files(root, allow_download=allow_download)
            snapshot = verify_snapshot(root)
            if not snapshot["converted"]:
                conversion = convert_model(root)
                if report is not None:
                    report({"conversion": conversion})
                snapshot = verify_snapshot(root)
            elif report is not None:
                report({"conversion": "converted file already present and digest-verified"})
            source = "converted from the manifest-verified source checkpoint"
        else:
            verify_converted(root)
            source = "converted file, pinned digest (source checkpoint not required)"
        import torch
        from safetensors.torch import load_file

        chosen = device or ("cuda" if torch.cuda.is_available() else "cpu")
        if chosen.startswith("cuda") and not torch.cuda.is_available():
            raise ValueError("device='cuda' requested but CUDA is not available")
        model = build_model()
        state = load_file(str(root / CONVERTED_WEIGHTS_NAME))
        if len(state) != STATE_TENSORS:
            raise ValueError(f"converted file holds {len(state)} tensors, expected {STATE_TENSORS}")
        model.load_state_dict(_expand_aliases(state), strict=True)
        n_params = sum(p.numel() for p in model.parameters())
        if n_params != PARAMETER_COUNT:
            raise ValueError(f"rebuilt model has {n_params} parameters, expected {PARAMETER_COUNT}")
        model.to(torch.device(chosen)).eval()
        for param in model.parameters():
            param.requires_grad_(False)
        return cls(model=model, device=chosen, weights_dir=root, source=source)

    # ---- forward ---------------------------------------------------------------------------------------

    def _to_tensor(self, images: Any) -> Any:
        """(B, H, W, 3) uint8 -> (B, 3, H, W) float32 in [-1, 1] on the device (upstream normalisation)."""
        import numpy as np
        import torch

        batch = torch.from_numpy(np.ascontiguousarray(images)).to(self.device).permute(0, 3, 1, 2).float()
        return (batch / 255.0 - 0.5) / 0.5

    def _matte_one(self, image: Any) -> Any:
        """One (H, W, 3) uint8 image -> (H, W) float32 matte at the input size (upstream resize rules)."""
        import torch
        import torch.nn.functional as F

        h, w = int(image.shape[0]), int(image.shape[1])
        rh, rw = _inference_size(h, w)
        batch = self._to_tensor(image[None])
        if (rh, rw) != (h, w):
            batch = F.interpolate(batch, size=(rh, rw), mode="area")
        with torch.inference_mode():
            _, _, matte = self.model(batch, True)
            if (rh, rw) != (h, w):
                matte = F.interpolate(matte, size=(h, w), mode="area")
        return matte[0, 0].clamp(0.0, 1.0).float().cpu().numpy()

    # ---- inference -------------------------------------------------------------------------------------

    def predict(self, records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
        """Matte images one at a time: per record the alpha matte (H, W) float32 in [0, 1] at the input size, the
        resized model input size, and the foreground fraction (mean alpha). The matte is the model's sigmoid output,
        not a calibrated probability."""
        checked = validate_dataset(records, min_records=1, require_labels=False)["records"]
        started = time.perf_counter()
        predictions = []
        for record in checked:
            matte = self._matte_one(record["image"])
            h, w = record["image"].shape[:2]
            predictions.append(
                {
                    "id": record["id"],
                    "alpha": matte,
                    "input_size": [int(h), int(w)],
                    "model_size": list(_inference_size(int(h), int(w))),
                    "foreground_fraction": round(float(matte.mean()), 4),
                }
            )
        return {
            "model": {"id": MODEL_ID, "revision": MODEL_REVISION, "key": MODEL_KEY, "adapted": self.adapter is not None},
            "output": "alpha matte in [0, 1] per pixel (sigmoid of the fusion branch; no threshold applied)",
            "predictions": predictions,
            "seconds": round(time.perf_counter() - started, 3),
        }

    def evaluate(self, records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
        """Matte errors on labelled records: MAD, MSE, SAD and the MAD over the trimap's unknown band, with the
        all-background and all-foreground constant mattes scored on the same pixels."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import constant_baselines, matting_metrics` removed — names are kernel globals defined by the carried modules

        checked = validate_dataset(records, min_records=1)["records"]
        started = time.perf_counter()
        result = self.predict(checked)
        predicted = [p["alpha"] for p in result["predictions"]]
        reference = [r["alpha"] for r in checked]
        unknown = [trimap_from_alpha(r["alpha"]) == 0.5 for r in checked]
        metrics = matting_metrics(predicted, reference, unknown=unknown)
        return {
            "n_records": len(checked),
            "metric": (
                "per-image alpha-matte MAD / MSE / SAD (÷1000) averaged over the records, and MAD over the trimap's unknown band"
            ),
            "model": {k: v for k, v in metrics.items() if k != "per_image"},
            "per_image": [{"id": r["id"], **row} for r, row in zip(checked, metrics["per_image"], strict=True)],
            "baselines": constant_baselines(reference, unknown=unknown),
            "adapted": self.adapter is not None,
            "seconds": round(time.perf_counter() - started, 3),
        }

    # ---- adaptation ------------------------------------------------------------------------------------

    def _trainable(self, mode: str) -> list[str]:
        if mode not in ADAPTATION_MODES:
            raise ValueError(f"trainable must be one of {ADAPTATION_MODES}")
        frozen = FROZEN_PREFIXES[mode]
        return sorted(
            name
            for name, _param in self.model.named_parameters()
            if not name.startswith(ALIAS_PREFIX) and not name.startswith(frozen)
        )

    def _losses(self, batch: Any, trimap: Any, gt_matte: Any) -> dict[str, Any]:
        """Upstream `supervised_training_iter` losses: semantic MSE against the blurred 1/16 matte, detail L1 in the
        unknown band, and matte L1 + compositional L1 (with the unknown band weighted 4×)."""
        import torch
        import torch.nn.functional as F

        pred_semantic, pred_detail, pred_matte = self.model(batch, False)
        boundaries = (trimap < 0.5) | (trimap > 0.5)
        gt_semantic = F.interpolate(gt_matte, scale_factor=1 / 16, mode="bilinear")
        kernel = torch.tensor(SEMANTIC_BLUR_KERNEL, dtype=gt_semantic.dtype, device=gt_semantic.device)[None, None]
        gt_semantic = F.conv2d(F.pad(gt_semantic, (1, 1, 1, 1), mode="reflect"), kernel)
        semantic_loss = LOSS_SCALES["semantic"] * F.mse_loss(pred_semantic, gt_semantic)
        pred_boundary_detail = torch.where(boundaries, trimap, pred_detail)
        gt_detail = torch.where(boundaries, trimap, gt_matte)
        detail_loss = LOSS_SCALES["detail"] * F.l1_loss(pred_boundary_detail, gt_detail)
        pred_boundary_matte = torch.where(boundaries, trimap, pred_matte)
        matte_l1 = F.l1_loss(pred_matte, gt_matte) + 4.0 * F.l1_loss(pred_boundary_matte, gt_matte)
        matte_comp = F.l1_loss(batch * pred_matte, batch * gt_matte) + 4.0 * F.l1_loss(
            batch * pred_boundary_matte, batch * gt_matte
        )
        matte_loss = LOSS_SCALES["matte"] * (matte_l1 + matte_comp)
        return {
            "semantic": semantic_loss,
            "detail": detail_loss,
            "matte": matte_loss,
            "total": semantic_loss + detail_loss + matte_loss,
        }

    def _batch_tensors(self, records: Sequence[Mapping[str, Any]]) -> tuple[Any, Any, Any]:
        import numpy as np
        import torch

        images = np.stack([r["image"] for r in records])
        mattes = np.stack([r["alpha"] for r in records]).astype(np.float32)
        trimaps = np.stack([r["trimap"] if "trimap" in r else trimap_from_alpha(r["alpha"]) for r in records]).astype(np.float32)
        return (
            self._to_tensor(images),
            torch.from_numpy(trimaps).to(self.device)[:, None],
            torch.from_numpy(mattes).to(self.device)[:, None],
        )

    def adapt(
        self,
        train: Sequence[Mapping[str, Any]],
        val: Sequence[Mapping[str, Any]] | None = None,
        *,
        epochs: int = 4,
        lr: float = 1e-4,
        batch_size: int = 4,
        trainable: str = "branches",
        seed: int = 0,
        progress: Callable[[dict[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded fine-tuning of the matting branches (`trainable="branches"`: everything except the MobileNetV2
        backbone; `"full"` unfreezes the backbone too) on labelled portrait/alpha pairs with the upstream supervised
        losses, Adam at a fixed learning rate, seeded horizontal flips, and BatchNorm statistics frozen. Epoch 0
        records the frozen model; the epoch with the lowest validation loss is kept."""
        if not isinstance(epochs, int) or not 1 <= epochs <= 50:
            raise ValueError("epochs must be an int in 1..50")
        if not (0.0 < lr <= 1e-2):
            raise ValueError("lr must be in (0, 1e-2]")
        if not isinstance(batch_size, int) or not 1 <= batch_size <= 16:
            raise ValueError("batch_size must be an int in 1..16")
        names = self._trainable(trainable)
        train_checked = validate_dataset(train)["records"]
        val_checked = validate_dataset(val, min_records=1)["records"] if val is not None else None
        for record in train_checked + (val_checked or []):  # trimaps once per record, not once per step
            record["trimap"] = trimap_from_alpha(record["alpha"])
        import numpy as np
        import torch

        torch.manual_seed(seed)
        started = time.perf_counter()
        model = self.model
        name_set = set(names)
        for name, param in model.named_parameters():
            param.requires_grad_(name in name_set)
        params = [p for n, p in model.named_parameters() if n in name_set]
        n_trainable = sum(p.numel() for p in params)
        optimiser = torch.optim.Adam(params, lr=lr, betas=(0.9, 0.99))
        rng = np.random.default_rng(seed)

        def val_loss() -> float | None:
            if val_checked is None:
                return None
            model.eval()
            losses = []
            with torch.inference_mode():
                for start in range(0, len(val_checked), batch_size):
                    batch, trimap, matte = self._batch_tensors(val_checked[start : start + batch_size])
                    losses.append(float(self._losses(batch, trimap, matte)["total"]))
            return sum(losses) / len(losses)

        initial_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in name_set}
        try:
            history: list[dict[str, Any]] = []
            entry: dict[str, Any] = {"epoch": 0, "train_loss": None, "val_loss": val_loss(), "note": "frozen model"}
            if val_checked is not None:
                entry["val"] = self.evaluate(val_checked)["model"]
            history.append(entry)
            best_val = entry["val_loss"] if entry["val_loss"] is not None else math.inf
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in name_set}
            best_epoch = 0
            if progress:
                progress(entry)
            n_steps = 0
            for epoch in range(1, epochs + 1):
                model.train()
                model.freeze_norm()  # BatchNorm statistics stay frozen (upstream SOC practice): small batches would corrupt them
                order = rng.permutation(len(train_checked)).tolist()
                losses = []
                parts = {"semantic": 0.0, "detail": 0.0, "matte": 0.0}
                for start in range(0, len(order), batch_size):
                    records = [train_checked[i] for i in order[start : start + batch_size]]
                    if rng.random() < 0.5:
                        records = [
                            {**r, "image": r["image"][:, ::-1], "alpha": r["alpha"][:, ::-1], "trimap": r["trimap"][:, ::-1]}
                            for r in records
                        ]
                    batch, trimap, matte = self._batch_tensors(records)
                    with torch.enable_grad():
                        loss = self._losses(batch, trimap, matte)
                        optimiser.zero_grad(set_to_none=True)
                        loss["total"].backward()
                        torch.nn.utils.clip_grad_norm_(params, 1.0)
                        optimiser.step()
                    losses.append(float(loss["total"].detach()))
                    for key in parts:
                        parts[key] += float(loss[key].detach())
                    n_steps += 1
                model.eval()
                entry = {
                    "epoch": epoch,
                    "train_loss": sum(losses) / len(losses),
                    "train_loss_parts": {k: round(v / len(losses), 4) for k, v in parts.items()},
                    "val_loss": val_loss(),
                }
                if val_checked is not None:
                    entry["val"] = self.evaluate(val_checked)["model"]
                history.append(entry)
                if progress:
                    progress(entry)
                if entry["val_loss"] is None or entry["val_loss"] < best_val:
                    best_val = entry["val_loss"] if entry["val_loss"] is not None else best_val
                    best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in name_set}
                    best_epoch = epoch
        except BaseException:
            # Transactional: a failure in training, validation or the progress callback leaves the model as it
            # was before adapt() (trained tensors restored), frozen, with no adapter attached.
            restore = dict(model.state_dict())
            restore.update(_expand_aliases(initial_state))
            model.load_state_dict(restore, strict=True)
            model.eval()
            for param in model.parameters():
                param.requires_grad_(False)
            self.adapter = None
            raise
        merged = dict(model.state_dict())
        merged.update(_expand_aliases(best_state))
        model.load_state_dict(merged, strict=True)
        model.eval()
        for param in model.parameters():
            param.requires_grad_(False)
        self.adapter = {
            "trainable": trainable,
            "trainable_names": names,
            "n_trainable": n_trainable,
            "n_total": sum(p.numel() for p in model.parameters()),
            "epochs": epochs,
            "best_epoch": best_epoch,
            "lr": lr,
            "batch_size": batch_size,
            "losses": (
                "upstream supervised losses: semantic (×10), detail (×10), matte L1 + compositional (×1); "
                f"trimap grown {TRIMAP_RADIUS} px from the fractional alpha"
            ),
            "augmentation": "seeded horizontal flips",
            "batchnorm": "running statistics frozen (eval mode) during adaptation",
            "precision": "float32",
            "n_train_records": len(train_checked),
            "n_steps": n_steps,
            "seed": seed,
            "history": history,
            "seconds": round(time.perf_counter() - started, 2),
        }
        return dict(self.adapter)

    # ---- artifacts -------------------------------------------------------------------------------------

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Write the adapted tensors as safetensors with a manifest."""
        if self.adapter is None:
            raise ValueError("nothing to save: call adapt() first")
        from safetensors.torch import save_file

        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = set(self.adapter["trainable_names"])
        tensors = {k: v.detach().cpu().contiguous() for k, v in self.model.state_dict().items() if k in names}
        weights_path = out / ARTIFACT_WEIGHTS_NAME
        save_file(tensors, str(weights_path), metadata={"format": "pt"})
        manifest = {
            "format": ARTIFACT_FORMAT,
            "format_version": ARTIFACT_FORMAT_VERSION,
            "base_model": {"id": MODEL_ID, "revision": MODEL_REVISION, "key": MODEL_KEY, "converted_sha256": CONVERTED_SHA256},
            "adapter": {k: v for k, v in self.adapter.items() if k not in ("history", "trainable_names")},
            "history": self.adapter["history"],
            "tensors": sorted(tensors),
            "files": [
                {"path": ARTIFACT_WEIGHTS_NAME, "bytes": weights_path.stat().st_size, "sha256": _sha256_file(weights_path)}
            ],
            "metadata": dict(metadata or {}),
        }
        (out / ARTIFACT_MANIFEST_NAME).write_text(json.dumps(manifest, indent=2), encoding="utf-8")
        return out

    @staticmethod
    def check_artifact_manifest(root: Path, manifest: Mapping[str, Any]) -> tuple[Path, str]:
        """Static checks on an adapter manifest, before any model or weights work: format and version, the pinned
        base and converted digest, exactly one weights entry named `adapter.safetensors` inside the artifact
        directory, and an adaptation mode that is one of the declared scopes. Returns the weights path and mode."""
        if manifest.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
        if manifest.get("format_version") != ARTIFACT_FORMAT_VERSION:
            raise ValueError(
                f"artifact format_version {manifest.get('format_version')!r} is not supported "
                f"(expected {ARTIFACT_FORMAT_VERSION!r})"
            )
        base = manifest.get("base_model", {})
        if (base.get("id"), base.get("revision")) != (MODEL_ID, MODEL_REVISION):
            raise ValueError("artifact was adapted from a different base model or revision")
        if base.get("converted_sha256") != CONVERTED_SHA256:
            raise ValueError("artifact records a different converted-base digest")
        files = manifest.get("files")
        if not isinstance(files, list) or len(files) != 1:
            raise ValueError("artifact manifest must list exactly one weights file")
        entry = files[0]
        if not isinstance(entry, Mapping) or entry.get("path") != ARTIFACT_WEIGHTS_NAME:
            raise ValueError(f"artifact weights file must be named {ARTIFACT_WEIGHTS_NAME!r}")
        weights_path = (root / entry["path"]).resolve()
        if weights_path.parent != root.resolve():
            raise ValueError("artifact weights file must sit inside the artifact directory")
        adapter = manifest.get("adapter")
        mode = adapter.get("trainable") if isinstance(adapter, Mapping) else None
        if mode not in ADAPTATION_MODES:
            raise ValueError(f"artifact adapter.trainable must be one of {ADAPTATION_MODES}")
        if not isinstance(manifest.get("tensors"), list):
            raise ValueError("artifact manifest must list its tensors")
        return weights_path, mode

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Verify an adapter's manifest, scope and digest, then overwrite exactly the tensors the scope allows."""
        root = Path(artifact_dir)
        manifest = json.loads((root / ARTIFACT_MANIFEST_NAME).read_text(encoding="utf-8"))
        weights_path, mode = self.check_artifact_manifest(root, manifest)
        expected = self._trainable(mode)
        if sorted(manifest["tensors"]) != expected:
            raise ValueError(
                f"artifact tensor list does not match the {len(expected)} tensors that trainable={mode!r} may change"
            )
        entry = manifest["files"][0]
        if _sha256_file(weights_path) != entry["sha256"] or weights_path.stat().st_size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: digest or size mismatch; refusing to load")
        from safetensors.torch import load_file

        tensors = load_file(str(weights_path))
        if sorted(tensors) != expected:
            raise ValueError("artifact tensor names differ from the validated manifest")
        state = self.model.state_dict()
        for key, value in tensors.items():
            if tuple(value.shape) != tuple(state[key].shape):
                raise ValueError(f"artifact tensor {key} has shape {tuple(value.shape)}, model has {tuple(state[key].shape)}")
        merged = dict(state)
        merged.update(_expand_aliases({k: v.to(state[k].device, state[k].dtype) for k, v in tensors.items()}))
        self.model.load_state_dict(merged, strict=True)
        self.model.eval()
        self.adapter = {**manifest["adapter"], "trainable_names": manifest["tensors"], "history": manifest.get("history", [])}
        return manifest

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
        require_source: bool = True,
    ) -> ModNetMattingPipeline:
        root = Path(artifact_dir)
        manifest = json.loads((root / ARTIFACT_MANIFEST_NAME).read_text(encoding="utf-8"))
        cls.check_artifact_manifest(root, manifest)
        pipeline = cls.from_pretrained(
            device=device, weights_dir=weights_dir, allow_download=allow_download, require_source=require_source
        )
        pipeline.load_artifact(artifact_dir)
        return pipeline

**Module 4/4:** `src/modnet_matting_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Tutorial data for the MODNet matting pipeline: a seeded synthetic portrait generator with exact alpha mattes (the
labelled sample dataset), four digest-pinned CC0 portrait photographs for label-free inference, the BYOD zip
loader, and the writers that put a sample pair on disk in the BYOD shape.

Why synthetic labels: no portrait-matting dataset with per-pixel alpha mattes is both permissively licensed and free
of personal-data concerns (P3M-10k, PPM-100 and AIM-500 are research-only), so the labelled records are drawn
figures — head, neck, shoulders, a hair cap and dozens of thin hair strands with fractional coverage — composited
over generated backgrounds with an exact alpha. They are out of MODNet's photographic training domain on purpose:
the frozen model's error on them and the adapted model's error are the tutorial's paired comparison, and the four
photographs (CC0, Pixabay via Wikimedia Commons) show the frozen and adapted models on real portraits without a
label. Everything here uses numpy and Pillow only; no model library is imported.
"""

from __future__ import annotations

import csv
import hashlib
import io
import json
import urllib.request
import zipfile
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

SAMPLE_SIZE = 512  # the tutorial's training resolution (MODNet's reference size)
SUPERSAMPLE = 2  # the figures are drawn at 1024 × 1024 and box-filtered down, which is where fractional alpha comes from
SAMPLE_COUNTS = {"train": 48, "validation": 12, "test": 20}
SAMPLE_SEEDS = {"train": 0, "validation": 1_000, "test": 2_000}  # disjoint seed ranges; ids carry the seed
SAMPLE_LABEL_SOURCE = "in-code synthetic portraits with exact alpha mattes (seeded numpy + Pillow renderer)"
DEFAULT_PORTRAIT_DIR = Path.cwd() / "weights" / "portraits"  # standalone rewrite (build_notebook.py): working-directory-relative
PORTRAIT_LICENSE = "CC0 1.0 (Pixabay photographs re-hosted on Wikimedia Commons)"
PORTRAIT_USER_AGENT = "modnet-matting-pipeline/0.1 (DIMER fleet tutorial by kurtvalcorza; digest-pinned fetch of four CC0 files)"
# Four CC0 stock portraits, pinned by URL, byte size and SHA-256; a re-upload under the same name changes the bytes
# and is refused. They are inference-only (no matte exists for them).
PORTRAIT_RECORDS: tuple[dict[str, Any], ...] = (
    {
        "id": "bearded-man-pipe",
        "url": "https://upload.wikimedia.org/wikipedia/commons/4/49/Bearded_man_smoking_pipe-3013924.jpg",
        "bytes": 4_702_901,
        "sha256": "aca3b45787b17c66309eaa10483d7c29d5689fbd82e773dce0f6fa299f735203",
        "title": "Bearded man smoking pipe (Pixabay 3013924)",
        "width": 5500,
        "height": 3667,
    },
    {
        "id": "portrait-in-hijab",
        "url": "https://upload.wikimedia.org/wikipedia/commons/9/9d/Portrait_in_hijab%2C_3064633.jpg",
        "bytes": 1_241_106,
        "sha256": "4cf790351645a6300c5529450eaa2da0581e8ed38e1171e60374049472ad190f",
        "title": "Portrait in hijab (Pixabay 3064633)",
        "width": 4256,
        "height": 2832,
    },
    {
        "id": "close-up-old-woman",
        "url": "https://upload.wikimedia.org/wikipedia/commons/8/80/Close-up_portrait_of_an_old_woman.jpg",
        "bytes": 1_607_542,
        "sha256": "03f80f34457f0f15c20251de8ca93fc9c97e379c6ebcee815ef64fe5fde03108",
        "title": "Close-up portrait of an old woman (Pixabay)",
        "width": 3008,
        "height": 2000,
    },
    {
        "id": "womans-close-portrait",
        "url": "https://upload.wikimedia.org/wikipedia/commons/7/70/Woman%27s_close_portrait%2C_3096664.jpg",
        "bytes": 1_718_396,
        "sha256": "eb77b3e953813c97978a70892d2646b88e21e227ae5bef9e6e81d310df67869b",
        "title": "Woman's close portrait (Pixabay 3096664)",
        "width": 3021,
        "height": 2351,
    },
)
PORTRAIT_MAX_SIDE = 1536  # the photographs are 3–5.5 k pixels wide; they are downscaled once on load (records say so)

# --------------------------------------------------------------------------------------------------
# synthetic portraits
# --------------------------------------------------------------------------------------------------

_SKIN = ((246, 219, 196), (226, 189, 158), (200, 152, 116), (168, 116, 84), (128, 80, 56), (92, 58, 40), (240, 205, 190))
_HAIR = (
    (28, 22, 20),
    (60, 40, 28),
    (110, 70, 40),
    (170, 120, 60),
    (210, 180, 120),
    (120, 120, 125),
    (230, 225, 220),
    (150, 40, 30),
)
_CLOTH = (
    (40, 60, 120),
    (150, 30, 40),
    (30, 110, 70),
    (230, 230, 225),
    (40, 40, 45),
    (200, 150, 40),
    (120, 60, 140),
    (90, 130, 180),
)
_BACK = (
    (200, 210, 225),
    (235, 225, 205),
    (120, 140, 160),
    (90, 100, 90),
    (220, 190, 170),
    (60, 65, 80),
    (170, 200, 180),
    (240, 240, 240),
)


def _jitter(rng: Any, colour: Sequence[int], spread: int = 18) -> tuple[int, int, int]:
    return tuple(int(min(255, max(0, c + rng.integers(-spread, spread + 1)))) for c in colour)  # type: ignore[return-value]


def _background(rng: Any, size: int) -> Any:
    """A gradient between two colours, a few soft blobs, optional wall/furniture rectangles, and grain."""
    import numpy as np
    from PIL import Image, ImageDraw, ImageFilter

    c1 = np.asarray(_jitter(rng, _BACK[int(rng.integers(len(_BACK)))]), dtype=np.float32)
    c2 = np.asarray(_jitter(rng, _BACK[int(rng.integers(len(_BACK)))]), dtype=np.float32)
    y, x = np.mgrid[0:size, 0:size].astype(np.float32) / size
    t = (0.6 * y + 0.4 * x) if rng.random() < 0.5 else y
    base = c1[None, None, :] * (1 - t[..., None]) + c2[None, None, :] * t[..., None]
    canvas = Image.fromarray(base.astype(np.uint8))
    draw = ImageDraw.Draw(canvas)
    for _ in range(int(rng.integers(0, 4))):  # furniture-like rectangles
        x0, y0 = rng.integers(0, size, 2)
        w, h = rng.integers(size // 8, size // 2, 2)
        draw.rectangle((int(x0), int(y0), int(x0 + w), int(y0 + h)), fill=_jitter(rng, _BACK[int(rng.integers(len(_BACK)))], 40))
    blobs = Image.new("RGB", (size, size), (0, 0, 0))
    bd = ImageDraw.Draw(blobs)
    for _ in range(int(rng.integers(2, 6))):
        cx, cy = rng.integers(0, size, 2)
        r = int(rng.integers(size // 6, size // 2))
        bd.ellipse(
            (int(cx - r), int(cy - r), int(cx + r), int(cy + r)), fill=_jitter(rng, _BACK[int(rng.integers(len(_BACK)))], 50)
        )
    blobs = blobs.filter(ImageFilter.GaussianBlur(size / 12))
    mixed = np.asarray(canvas, dtype=np.float32) * 0.65 + np.asarray(blobs, dtype=np.float32) * 0.35
    grain = rng.normal(0, 4.0, mixed.shape).astype(np.float32)
    return np.clip(mixed + grain, 0, 255)


def _figure(rng: Any, size: int) -> tuple[Any, Any, dict[str, Any]]:
    """Draw one portrait figure at `size` × `size`: returns the foreground RGB (float32), the alpha (float32 in
    [0, 1], still at drawing resolution) and a record of what was drawn."""
    import numpy as np
    from PIL import Image, ImageDraw, ImageFilter

    fg = Image.new("RGB", (size, size), (0, 0, 0))
    alpha = Image.new("L", (size, size), 0)
    fd, ad = ImageDraw.Draw(fg), ImageDraw.Draw(alpha)
    skin = _jitter(rng, _SKIN[int(rng.integers(len(_SKIN)))], 10)
    hair = _jitter(rng, _HAIR[int(rng.integers(len(_HAIR)))], 12)
    cloth = _jitter(rng, _CLOTH[int(rng.integers(len(_CLOTH)))], 20)
    cx = size * float(rng.uniform(0.38, 0.62))
    cy = size * float(rng.uniform(0.34, 0.48))
    rx = size * float(rng.uniform(0.12, 0.18))
    ry = rx * float(rng.uniform(1.2, 1.4))
    shoulder_y = cy + ry + size * float(rng.uniform(0.10, 0.18))
    shoulder_w = size * float(rng.uniform(0.28, 0.42))
    neck_w = rx * float(rng.uniform(0.55, 0.8))
    # torso: a rounded trapezoid from the shoulders to the bottom edge
    torso = [
        (cx - shoulder_w, size + 10),
        (cx - shoulder_w, shoulder_y + shoulder_w * 0.35),
        (cx - shoulder_w * 0.6, shoulder_y),
        (cx + shoulder_w * 0.6, shoulder_y),
        (cx + shoulder_w, shoulder_y + shoulder_w * 0.35),
        (cx + shoulder_w, size + 10),
    ]
    for canvas, fill in ((fd, cloth), (ad, 255)):
        canvas.polygon(torso, fill=fill)
        canvas.ellipse(
            (cx - shoulder_w, shoulder_y - shoulder_w * 0.1, cx - shoulder_w * 0.3, shoulder_y + shoulder_w * 0.6), fill=fill
        )
        canvas.ellipse(
            (cx + shoulder_w * 0.3, shoulder_y - shoulder_w * 0.1, cx + shoulder_w, shoulder_y + shoulder_w * 0.6), fill=fill
        )
    # collar / neckline in the skin colour, then the neck
    for canvas, fill in ((fd, skin), (ad, 255)):
        canvas.rectangle((cx - neck_w, cy + ry * 0.6, cx + neck_w, shoulder_y + neck_w * 0.4), fill=fill)
        canvas.ellipse((cx - neck_w * 1.3, shoulder_y - neck_w * 0.5, cx + neck_w * 1.3, shoulder_y + neck_w * 0.8), fill=fill)
    # ears, head
    ear = rx * 0.22
    for canvas, fill in ((fd, skin), (ad, 255)):
        canvas.ellipse((cx - rx - ear, cy - ear * 1.3, cx - rx + ear, cy + ear * 1.3), fill=fill)
        canvas.ellipse((cx + rx - ear, cy - ear * 1.3, cx + rx + ear, cy + ear * 1.3), fill=fill)
        canvas.ellipse((cx - rx, cy - ry, cx + rx, cy + ry), fill=fill)
    # hair cap over the top of the head, optionally long hair falling beside the neck
    long_hair = rng.random() < 0.5
    cap_top = cy - ry * float(rng.uniform(1.05, 1.25))
    for canvas, fill in ((fd, hair), (ad, 255)):
        canvas.chord((cx - rx * 1.08, cap_top, cx + rx * 1.08, cy + ry * 0.35), 180, 360, fill=fill)
        if long_hair:
            canvas.polygon(
                [
                    (cx - rx * 1.05, cy - ry * 0.2),
                    (cx - rx * 1.25, shoulder_y + rx * 0.4),
                    (cx - rx * 0.9, shoulder_y + rx * 0.3),
                    (cx - rx * 0.95, cy),
                ],
                fill=fill,
            )
            canvas.polygon(
                [
                    (cx + rx * 1.05, cy - ry * 0.2),
                    (cx + rx * 1.25, shoulder_y + rx * 0.4),
                    (cx + rx * 0.9, shoulder_y + rx * 0.3),
                    (cx + rx * 0.95, cy),
                ],
                fill=fill,
            )
    # face features are drawn on the foreground only (they do not change the alpha)
    eye_y = cy - ry * 0.12
    for sx in (-1, 1):
        ex = cx + sx * rx * 0.42
        fd.ellipse((ex - rx * 0.16, eye_y - rx * 0.09, ex + rx * 0.16, eye_y + rx * 0.09), fill=(250, 250, 250))
        fd.ellipse((ex - rx * 0.07, eye_y - rx * 0.07, ex + rx * 0.07, eye_y + rx * 0.07), fill=_jitter(rng, (40, 30, 30), 20))
        fd.line((ex - rx * 0.2, eye_y - rx * 0.24, ex + rx * 0.2, eye_y - rx * 0.26), fill=hair, width=max(2, int(rx * 0.05)))
    fd.line((cx, cy - ry * 0.05, cx - rx * 0.08, cy + ry * 0.28), fill=_jitter(rng, skin, 30), width=max(2, int(rx * 0.04)))
    fd.arc(
        (cx - rx * 0.3, cy + ry * 0.35, cx + rx * 0.3, cy + ry * 0.62),
        10,
        170,
        fill=_jitter(rng, (150, 70, 80), 20),
        width=max(2, int(rx * 0.06)),
    )
    glasses = rng.random() < 0.3
    if glasses:
        for sx in (-1, 1):
            ex = cx + sx * rx * 0.42
            fd.rectangle(
                (ex - rx * 0.24, eye_y - rx * 0.16, ex + rx * 0.24, eye_y + rx * 0.16),
                outline=(30, 30, 30),
                width=max(2, int(rx * 0.04)),
            )
    # hair strands: thin lines from the hair boundary outward, with a partial-coverage brush, so the alpha at the
    # silhouette is fractional after the box filter — the part of a matte that matting exists for
    n_strands = int(rng.integers(40, 120))
    strand_sigma = float(rng.uniform(0.0, 1.2))
    for _ in range(n_strands):
        angle = float(rng.uniform(np.pi * 1.05, np.pi * 1.95))  # over the cap (upper half)
        if long_hair and rng.random() < 0.4:
            side = -1.0 if rng.random() < 0.5 else 1.0  # flyaway hair off the outer edge of the long-hair strips
            angle = float(rng.uniform(-0.35 * np.pi, 0.35 * np.pi)) + (np.pi if side < 0 else 0.0)
            x0, y0 = cx + side * rx * 1.2, float(rng.uniform(cy, shoulder_y + rx * 0.3))
        else:
            x0, y0 = cx + np.cos(angle) * rx * 1.05, cy + ry * 0.2 + np.sin(angle) * ry * 1.0
        length = size * float(rng.uniform(0.015, 0.08))
        bend = float(rng.uniform(-0.6, 0.6))
        pts = [(x0, y0)]
        for k in (0.5, 1.0):
            a = angle + bend * k
            pts.append((x0 + np.cos(a) * length * k, y0 + np.sin(a) * length * k))
        width = int(rng.integers(1, 4))
        coverage = int(255 * float(rng.uniform(0.35, 1.0)))
        fd.line(pts, fill=hair, width=width)
        ad.line(pts, fill=coverage, width=width)
    if strand_sigma > 0.2:
        alpha = alpha.filter(ImageFilter.GaussianBlur(strand_sigma))
    record = {
        "skin": skin,
        "hair": hair,
        "cloth": cloth,
        "head_center": (round(cx / size, 3), round(cy / size, 3)),
        "long_hair": long_hair,
        "glasses": glasses,
        "strands": n_strands,
    }
    return np.asarray(fg, dtype=np.float32), np.asarray(alpha, dtype=np.float32) / 255.0, record


def render_portrait(seed: int, *, size: int = SAMPLE_SIZE) -> dict[str, Any]:
    """One synthetic portrait: `{id, image (size, size, 3) uint8, alpha (size, size) float32 in [0, 1], meta}`,
    deterministic in `seed`. Drawn at `SUPERSAMPLE` × size and box-filtered down, so the alpha is fractional
    along every hair strand and silhouette edge; composited as alpha · foreground + (1 − alpha) · background."""
    import numpy as np
    from PIL import Image

    rng = np.random.default_rng(seed)
    big = size * SUPERSAMPLE
    background = _background(rng, big)
    fg, alpha, meta = _figure(rng, big)
    composite = alpha[..., None] * fg + (1.0 - alpha[..., None]) * background
    image = Image.fromarray(np.clip(composite, 0, 255).astype(np.uint8)).resize((size, size), Image.Resampling.BOX)
    matte = Image.fromarray(np.clip(alpha * 255.0, 0, 255).astype(np.uint8)).resize((size, size), Image.Resampling.BOX)
    return {
        "id": f"portrait-{seed:05d}",
        "image": np.asarray(image, dtype=np.uint8),
        "alpha": np.asarray(matte, dtype=np.float32) / 255.0,
        "meta": {"seed": seed, **meta},
    }


def sample_dataset(*, counts: Mapping[str, int] | None = None, size: int = SAMPLE_SIZE) -> dict[str, list[dict[str, Any]]]:
    """The tutorial's labelled splits (48 / 12 / 20 by default), each drawn from its own seed range."""
    counts = dict(SAMPLE_COUNTS if counts is None else counts)
    out: dict[str, list[dict[str, Any]]] = {}
    for split, n in counts.items():
        if split not in SAMPLE_SEEDS:
            raise ValueError(f"unknown split {split!r}; expected {sorted(SAMPLE_SEEDS)}")
        if not isinstance(n, int) or not 1 <= n <= 500:
            raise ValueError(f"{split}: count must be an int in 1..500")
        base = SAMPLE_SEEDS[split]
        out[split] = [render_portrait(base + i, size=size) for i in range(n)]
    return out


def split_dataset(
    records: Sequence[Mapping[str, Any]], *, seed: int = 0, fractions: tuple[float, float] = (0.7, 0.15)
) -> dict[str, list[dict[str, Any]]]:
    """Seeded shuffle of user records into train / validation / test (validation and test hold at least one
    record each when there are three or more records)."""
    import numpy as np

    if len(records) < 3:
        raise ValueError("at least three labelled records are needed to form train / validation / test splits")
    order = np.random.default_rng(seed).permutation(len(records)).tolist()
    n_train = max(1, int(round(len(records) * fractions[0])))
    n_val = max(1, int(round(len(records) * fractions[1])))
    if n_train + n_val >= len(records):
        n_train = len(records) - n_val - 1
    shuffled = [dict(records[i]) for i in order]
    return {"train": shuffled[:n_train], "validation": shuffled[n_train : n_train + n_val], "test": shuffled[n_train + n_val :]}


def check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Refuse a record (by id, and by image bytes) present in more than one split."""
    seen_ids: dict[str, str] = {}
    seen_digests: dict[str, str] = {}
    for split, records in splits.items():
        for record in records:
            rid = str(record["id"])
            if rid in seen_ids and seen_ids[rid] != split:
                raise ValueError(f"record {rid!r} is in both {seen_ids[rid]!r} and {split!r}")
            seen_ids[rid] = split
            digest = hashlib.sha256(_image_bytes(record["image"])).hexdigest()
            if digest in seen_digests and seen_digests[digest] != split:
                raise ValueError(f"record {rid!r} duplicates an image in {seen_digests[digest]!r}")
            seen_digests[digest] = split
    return {"disjoint": True, "n_ids": len(seen_ids)}


def _image_bytes(image: Any) -> bytes:
    import numpy as np

    if isinstance(image, str | Path):
        return Path(image).read_bytes()
    return np.ascontiguousarray(np.asarray(image)).tobytes()


# --------------------------------------------------------------------------------------------------
# pinned photographs (inference only)
# --------------------------------------------------------------------------------------------------


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def fetch_portraits(cache_dir: str | Path | None = None, *, fetcher: Any | None = None) -> list[dict[str, Any]]:
    """Fetch the four pinned CC0 photographs into `cache_dir` (default `weights/portraits/`, git-ignored), each
    refused on a byte-size or SHA-256 mismatch, and return inference records `{id, image, title, source}`. The
    photographs are downscaled on load to at most `PORTRAIT_MAX_SIDE` pixels on the long side (recorded per
    record) — MODNet resizes to 512 on the short side anyway, and the originals are 3–5.5 k pixels wide."""
    import numpy as np
    from PIL import Image

    root = Path(cache_dir) if cache_dir is not None else DEFAULT_PORTRAIT_DIR
    root.mkdir(parents=True, exist_ok=True)
    records = []
    for pin in PORTRAIT_RECORDS:
        target = root / f"{pin['id']}.jpg"
        if not (
            target.is_file() and target.stat().st_size == pin["bytes"] and _sha256_bytes(target.read_bytes()) == pin["sha256"]
        ):
            data = (fetcher or _http_get)(pin["url"])
            if len(data) != pin["bytes"] or _sha256_bytes(data) != pin["sha256"]:
                raise ValueError(f"{pin['id']}: downloaded bytes do not match the pinned size/SHA-256; refusing")
            target.write_bytes(data)
        with Image.open(target) as im:
            im = im.convert("RGB")
            original = im.size
            if max(im.size) > PORTRAIT_MAX_SIDE:
                scale = PORTRAIT_MAX_SIDE / max(im.size)
                im = im.resize((max(1, round(im.width * scale)), max(1, round(im.height * scale))), Image.Resampling.LANCZOS)
            image = np.asarray(im, dtype=np.uint8)
        records.append(
            {
                "id": pin["id"],
                "image": image,
                "title": pin["title"],
                "source": pin["url"],
                "license": PORTRAIT_LICENSE,
                "original_size": list(original),
                "loaded_size": [int(image.shape[1]), int(image.shape[0])],
            }
        )
    return records


def _http_get(url: str) -> bytes:
    request = urllib.request.Request(url, headers={"User-Agent": PORTRAIT_USER_AGENT})
    with urllib.request.urlopen(request, timeout=180) as response:  # noqa: S310 - pinned https URL, digest-verified
        return response.read()


# --------------------------------------------------------------------------------------------------
# BYOD: zip of image / alpha pairs + pairs.csv
# --------------------------------------------------------------------------------------------------

BYOD_MAX_MEMBERS = 2_000
BYOD_MAX_BYTES = 2_000_000_000


def load_byod_dataset(path: str | Path, *, size: int = SAMPLE_SIZE) -> list[dict[str, Any]]:
    """Read a zip holding `pairs.csv` (columns `id`, `image`, `alpha`) beside RGB images and single-channel alpha
    PNGs (0 = background, 255 = foreground). Members are read through the archive API by the names the CSV lists
    (no `extractall`, no paths from the archive are written anywhere); every pair is resized to `size` × `size`
    (recorded in the record) so it fits the training contract."""
    import numpy as np
    from PIL import Image

    zip_path = Path(path)
    if not zip_path.is_file():
        raise FileNotFoundError(f"BYOD zip not found: {zip_path}")
    if zip_path.stat().st_size > BYOD_MAX_BYTES:
        raise ValueError(f"BYOD zip larger than {BYOD_MAX_BYTES} bytes")
    with zipfile.ZipFile(zip_path) as archive:
        names = {info.filename: info for info in archive.infolist() if not info.is_dir()}
        if len(names) > BYOD_MAX_MEMBERS:
            raise ValueError(f"BYOD zip holds more than {BYOD_MAX_MEMBERS} members")
        csv_name = next((n for n in names if n.endswith("pairs.csv")), None)
        if csv_name is None:
            raise ValueError("BYOD zip must contain pairs.csv")
        prefix = csv_name[: -len("pairs.csv")]
        rows = list(csv.DictReader(io.StringIO(archive.read(csv_name).decode("utf-8"))))
        if not rows or not {"id", "image", "alpha"} <= set(rows[0]):
            raise ValueError("pairs.csv must have the columns id, image, alpha")
        records = []
        for row in rows:
            image_name, alpha_name = prefix + row["image"], prefix + row["alpha"]
            for name in (image_name, alpha_name):
                if name not in names:
                    raise ValueError(f"pairs.csv names a member that is not in the archive: {name}")
            with Image.open(io.BytesIO(archive.read(image_name))) as im:
                original = im.size
                image = np.asarray(im.convert("RGB").resize((size, size), Image.Resampling.LANCZOS), dtype=np.uint8)
            with Image.open(io.BytesIO(archive.read(alpha_name))) as am:
                alpha = np.asarray(am.convert("L").resize((size, size), Image.Resampling.LANCZOS), dtype=np.float32) / 255.0
            records.append(
                {"id": str(row["id"]), "image": image, "alpha": alpha, "source": row["image"], "original_size": list(original)}
            )
    return records


def write_sample_pair(record: Mapping[str, Any], image_path: str | Path, alpha_path: str | Path) -> dict[str, Any]:
    """Write one record as the BYOD pair shape: an RGB PNG and an 8-bit alpha PNG."""
    import numpy as np
    from PIL import Image

    Path(image_path).parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(np.asarray(record["image"], dtype=np.uint8)).save(image_path)
    Image.fromarray(np.clip(np.asarray(record["alpha"], dtype=np.float32) * 255.0 + 0.5, 0, 255).astype(np.uint8)).save(
        alpha_path
    )
    return {"image": str(image_path), "alpha": str(alpha_path), "id": record["id"]}


def write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    """The `pairs.csv` a BYOD zip is expected to carry, listing the records by id (image/alpha names by convention)."""
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", newline="", encoding="utf-8") as fh:
        writer = csv.writer(fh)
        writer.writerow(["id", "image", "alpha"])
        for record in records:
            writer.writerow([record["id"], f"{record['id']}.png", f"{record['id']}_alpha.png"])
    return out


def dataset_manifest(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Validate every split with the pipeline's checker, refuse overlaps, and record per-split facts and a digest
    (the pipeline module is imported lazily so this module stays model-free)."""
    pass  # standalone rewrite (build_notebook.py): `from .pipeline import validate_dataset` removed — names are kernel globals defined by the carried modules

    reports = {name: validate_dataset(records, min_records=1) for name, records in splits.items()}
    disjoint = check_split_disjoint(splits)
    payload = json.dumps({name: report["digest"] for name, report in reports.items()}, sort_keys=True)
    return {
        "splits": {name: {k: v for k, v in report.items() if k != "records"} for name, report in reports.items()},
        "disjoint": disjoint["disjoint"],
        "digest": hashlib.sha256(payload.encode("utf-8")).hexdigest(),
    }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `1`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `71aca6d04ed0…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `ModNetMattingPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, device=('cuda' if torch.cuda.is_available() else 'cpu'), report=print)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "modnet-photographic-portrait-matting",
  "modelId": "XM5354/Modnet_models",
  "revision": "71aca6d04ed0267b4b12bde776868f2b9fb1d06f",
  "files": [
    {
      "path": "modnet_photographic_portrait_matting.ckpt",
      "bytes": 26255603,
      "sha256": "7c22235f0925deba15d4d63e53afcb654c47055bbcd98f56e393ab2584007ed8"
    }
  ],
  "totalBytes": 26255603
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = ModNetMattingPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, device=('cuda' if torch.cuda.is_available() else 'cpu'), report=print)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Sample portraits, validation and roles

The default labelled dataset is rendered here, in the kernel, by `sample_dataset`: 48 training, 12 validation and 20 test portraits from three disjoint seed ranges, each drawn at 1024 × 1024 and box-filtered to 512 × 512 so that every hair strand and silhouette edge carries fractional alpha, composited as alpha · figure + (1 − alpha) · background. `dataset_manifest` validates every split with the same checker the model path uses, refuses a portrait present in two splits and records a digest; `fetch_portraits` downloads the four CC0 photographs (each refused on a size or digest mismatch) and downscales them once for the record.

Look for: 48 / 12 / 20 records with foreground fractions around 0.34 and 2 % fractional-alpha pixels, a written sample pair (`outputs/modnet_matting_sample_portrait.png` + `_sample_alpha.png`, the BYOD shape), the four photographs with their original and loaded sizes, and three refusal probes — an alpha outside [0, 1], a labelled record that is not 512 × 512, and a dataset whose alphas are all background — each rejected before the model runs.

In [ ]:
import json
import os
from pathlib import Path

import numpy as np

USE_BYOD = False  # @param {type:"boolean"}

os.makedirs('outputs', exist_ok=True)
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    file_name, payload = next(iter(uploaded.items()))
    byod_path = Path('work') / file_name
    byod_path.parent.mkdir(parents=True, exist_ok=True)
    byod_path.write_bytes(payload)
    splits = split_dataset(load_byod_dataset(byod_path), seed=0)
    data_source = 'BYOD (' + file_name + ')'
else:
    splits = sample_dataset()
    data_source = SAMPLE_LABEL_SOURCE
train_records, val_records, test_records = splits['train'], splits['validation'], splits['test']

dataset_report = dataset_manifest({'train': train_records, 'validation': val_records, 'test': test_records})
print({'data_source': data_source, 'splits': {k: v['n_records'] for k, v in dataset_report['splits'].items()}, 'disjoint': dataset_report['disjoint'], 'digest': dataset_report['digest'][:16] + '...'})
for name, part in dataset_report['splits'].items():
    print({name: {'foreground_fraction': part['foreground_fraction'], 'fractional_alpha_fraction': part['fractional_alpha_fraction'], 'sizes': part['sizes']}})
print({'first_test_record': validate_inputs(test_records[0])})
sample_pair = write_sample_pair(test_records[0], 'outputs/modnet_matting_sample_portrait.png', 'outputs/modnet_matting_sample_alpha.png')
print({'sample_pair': sample_pair, 'pairs_csv': str(write_dataset_csv(test_records, 'outputs/modnet_matting_sample_pairs.csv'))})

portraits = fetch_portraits(cache_dir='weights/portraits')
for record in portraits:
    print({'photograph': record['id'], 'title': record['title'], 'original_size': record['original_size'], 'loaded_size': record['loaded_size'], 'license': record['license']})

print({'validation': INPUT_SCHEMA['validation']})
probes = {
    'alpha outside [0, 1]': [{**test_records[0], 'alpha': test_records[0]['alpha'] * 1.5}, *test_records[1:4]],
    'labelled record not 512 x 512': [{**test_records[0], 'image': test_records[0]['image'][:256], 'alpha': test_records[0]['alpha'][:256]}, *test_records[1:4]],
    'all-background alphas': [{**r, 'alpha': np.zeros_like(r['alpha'])} for r in test_records[:4]],
}
for name, records in probes.items():
    try:
        validate_dataset(records)
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

## 5. The frozen model: constant baselines, held-out errors and the photographs

`pipe.predict` normalises each image to [−1, 1], resizes it as the upstream inference script does (short side 512, both sides multiples of 32), runs the three branches and returns the fusion branch's sigmoid output as the matte at the input size — the model's output, not a calibrated probability. `pipe.evaluate` scores labelled records per image and averages: MAD and MSE over every pixel, SAD (the summed absolute difference ÷ 1000), and the MAD over the trimap's unknown band (every fractional-alpha pixel grown by 8 pixels — the part of a matte that matting exists for); the two **constant baselines** — every pixel background, every pixel subject — are scored on the same references, so all-background MAD equals the foreground fraction.

Look for: a frozen test MAD near 0.08 (in the build record 0.079, against 0.341 for all-background and 0.659 for all-foreground) — the photographic model finds the drawn heads but drops parts of the drawn clothing and misses strands — and, on the four photographs, mattes whose foreground fractions run from about 0.40 (the man with the pipe against a dark background) to 0.84 (a face filling the frame); the frozen mattes are written to `outputs/` as PNGs beside a cut-out on white. These are sample-sanity numbers on 20 and 12 drawn portraits, not a benchmark.

In [ ]:
import time

from PIL import Image

def write_matte(record, alpha, tag):
    matte = Image.fromarray(np.clip(alpha * 255.0 + 0.5, 0, 255).astype(np.uint8))
    matte.save(f'outputs/modnet_matting_matte_' + tag + '_' + record['id'] + '.png')
    cutout = (alpha[..., None] * record['image'].astype(np.float32) + (1.0 - alpha[..., None]) * 255.0)
    Image.fromarray(np.clip(cutout + 0.5, 0, 255).astype(np.uint8)).save(f'outputs/modnet_matting_cutout_' + tag + '_' + record['id'] + '.png')

t0 = time.perf_counter()
frozen_test = pipe.evaluate(test_records)
frozen_val = pipe.evaluate(val_records)
print({'seconds': round(time.perf_counter() - t0, 1), 'metric': frozen_test['metric']})
print({'baselines_test': {k: {m: v[m] for m in ('mad', 'mse', 'sad', 'mad_unknown')} for k, v in frozen_test['baselines'].items()}})
print({'frozen_test': frozen_test['model']})
print({'frozen_validation': frozen_val['model']})
for row in frozen_test['per_image'][:5]:
    print({'portrait': row['id'], 'mad': row['mad'], 'mad_unknown': row['mad_unknown']})
frozen_photos = pipe.predict(portraits)
for record, pred in zip(portraits, frozen_photos['predictions']):
    write_matte(record, pred['alpha'], 'frozen')
    print({'photograph': record['id'], 'model_size': pred['model_size'], 'foreground_fraction': pred['foreground_fraction'], 'note': 'no label; sanity check'})
print({'output': frozen_photos['output'], 'alpha_shape': frozen_photos['predictions'][0]['alpha'].shape, 'seconds': frozen_photos['seconds']})

## 6. Bounded fine-tuning of the matting branches

`pipe.adapt` trains the low-resolution branch's SE block and convolutions, the high-resolution branch and the fusion branch (4.26 M parameters — 66 % of the model) and nothing else: the MobileNetV2 backbone is frozen (no gradient is stored for it), and every BatchNorm layer keeps its running statistics, as the upstream adaptation code does. Each step takes four portraits with a seeded horizontal flip, derives the trimap from the reference alpha, and minimises the upstream supervised objective — the semantic loss (MSE against the blurred 1/16 matte, ×10), the detail loss (L1 in the unknown band, ×10) and the matte loss (L1 plus a compositional L1 with the unknown band weighted 4×) — with Adam at a small fixed learning rate and gradient-norm clipping. Epoch 0 records the frozen model's validation loss and metrics; the epoch with the lowest validation loss is kept.

Watch the validation loss: in the build record it fell from 0.80 to about 0.06 by epoch 4 and drifted afterwards, and the validation MAD from 0.064 to about 0.004. Six epochs (72 steps) take about 15 s on a T4 and about 3 minutes on a laptop CPU. `TRAINABLE = 'full'` also unfreezes the backbone (6.49 M parameters).

In [ ]:
EPOCHS = 6  # @param {type:"integer"}
LEARNING_RATE = 1e-4  # @param {type:"number"}
BATCH_SIZE = 4  # @param {type:"integer"}
TRAINABLE = 'branches'  # @param ["branches", "full"]

def report(entry):
    row = {'epoch': entry['epoch'], 'train_loss': None if entry['train_loss'] is None else round(entry['train_loss'], 4), 'val_loss': round(entry['val_loss'], 4)}
    if 'train_loss_parts' in entry:
        row['parts'] = entry['train_loss_parts']
    if 'val' in entry:
        row['val_mad'] = entry['val']['mad']
        row['val_mad_unknown'] = entry['val']['mad_unknown']
    if 'note' in entry:
        row['note'] = entry['note']
    print(row)

t0 = time.perf_counter()
adapt_result = pipe.adapt(train_records, val_records, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, trainable=TRAINABLE, progress=report)
adapt_seconds = round(time.perf_counter() - t0, 1)
print({'trainable_parameters': adapt_result['n_trainable'], 'total_parameters': adapt_result['n_total'], 'steps': adapt_result['n_steps'], 'best_epoch': adapt_result['best_epoch'], 'losses': adapt_result['losses'], 'batchnorm': adapt_result['batchnorm'], 'seconds': adapt_seconds})

## 7. Held-out evaluation: the paired comparison

The test portraits were never used for training or epoch selection (they come from their own seed range). The adapted model is scored exactly as the frozen model was in Section 5, and the table puts the baselines, the frozen and the adapted numbers side by side. The cell asserts what the procedure guarantees — the kept epoch's validation loss is no higher than the frozen model's, and re-scoring the validation portraits reproduces the kept epoch's MAD within 0.001 — and it also asserts that the adapted test MAD is below the frozen one: on drawn portraits the domain shift is large enough that the build record moved the test MAD from 0.079 to 0.004 (unknown-band MAD 0.085 → 0.025) on every hyperparameter probe, so a failure here is a finding, not noise. The size of the gain, and its reading — the model learned the drawing style, on 48 portraits, with one seed and no dispersion estimate — is not a quality claim about photographs; with your own portraits the gap between frozen and adapted is the number to watch.

In [ ]:
adapted_test = pipe.evaluate(test_records)
adapted_val = pipe.evaluate(val_records)
comparison = {}
for key in ('mad', 'mse', 'sad', 'mad_unknown'):
    comparison[key] = {'all_background': frozen_test['baselines']['all_background'][key], 'all_foreground': frozen_test['baselines']['all_foreground'][key], 'frozen': frozen_test['model'][key], 'adapted': adapted_test['model'][key]}
for key, row in comparison.items():
    print({key: row})
best = adapt_result['history'][adapt_result['best_epoch']]
print({'validation_mad': {'frozen': frozen_val['model']['mad'], 'adapted': adapted_val['model']['mad']}, 'validation_loss': {'frozen': adapt_result['history'][0]['val_loss'], 'kept_epoch': best['val_loss']}})
evaluation_report = {
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'key': MODEL_KEY},
    'data_source': data_source,
    'dataset': dataset_report,
    'frozen': {'test': frozen_test, 'validation': frozen_val},
    'adapted': {'test': adapted_test, 'validation': adapted_val},
    'comparison': comparison,
    'adaptation': {k: v for k, v in adapt_result.items() if k not in ('history', 'trainable_names')},
    'history': adapt_result['history'],
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/modnet_matting_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report, f, indent=2)
assert best['val_loss'] <= adapt_result['history'][0]['val_loss']
assert abs(adapted_val['model']['mad'] - best['val']['mad']) < 1e-3
assert adapted_test['model']['mad'] < frozen_test['model']['mad']
print({'report': 'outputs/modnet_matting_evaluation_report.json'})

## 8. The photographs again, artifact export and fresh reload

The adapted model mattes the same four photographs; they carry no label, so the comparison is the mean absolute difference between the frozen and the adapted matte per photograph and the foreground fractions side by side — a sanity check on whether learning the drawing style moved the model on real portraits (the build record: differences of 0.004–0.05; on the man with the pipe the adapted matte took in dark background beside the body, the silhouettes themselves stayed), not an evaluation. The adapted mattes and cut-outs are written next to the frozen ones.

`pipe.save_artifact` writes the trained tensors (about 17 MB) as `adapter.safetensors`, with a `manifest.json` recording the artifact format, the base model id and revision, the digest of the converted base file, the adaptation scope, the tensor names, the file size and SHA-256, the training configuration and the epoch history (OUT8). `ModNetMattingPipeline.from_artifact` re-verifies the base file, checks the artifact manifest, scope and digest **before** deserialising, rebuilds the model and overlays the tensors — a fresh object from files, not the in-memory model (VER2). The cell asserts the same held-out MAD within 0.0001 and mattes within 0.001 (VER4).

In [ ]:
import platform
import shutil

adapted_photos = pipe.predict(portraits)
photo_drift = []
for record, before, after in zip(portraits, frozen_photos['predictions'], adapted_photos['predictions']):
    write_matte(record, after['alpha'], 'adapted')
    drift = {'photograph': record['id'], 'foreground_fraction_frozen': before['foreground_fraction'], 'foreground_fraction_adapted': after['foreground_fraction'], 'mean_abs_matte_difference': round(float(np.abs(after['alpha'] - before['alpha']).mean()), 4)}
    photo_drift.append(drift)
    print({**drift, 'note': 'no label; sanity check'})
with open('outputs/modnet_matting_predictions.json', 'w', encoding='utf-8') as f:
    json.dump({'model': adapted_photos['model'], 'output': adapted_photos['output'], 'photographs': [{'id': r['id'], 'title': r['title'], 'source': r['source'], 'license': r['license']} for r in portraits], 'frozen': [{k: v for k, v in p.items() if k != 'alpha'} for p in frozen_photos['predictions']], 'adapted': [{k: v for k, v in p.items() if k != 'alpha'} for p in adapted_photos['predictions']], 'drift': photo_drift}, f, indent=2)

artifact_dir = Path('outputs/modnet_matting_adapter')
shutil.rmtree(artifact_dir, ignore_errors=True)
pipe.save_artifact(artifact_dir, metadata={'tutorial': 'modnet_matting', 'data_source': data_source})
artifact_manifest = json.loads((artifact_dir / 'manifest.json').read_text(encoding='utf-8'))
print({'artifact': str(artifact_dir), 'format': artifact_manifest['format'], 'trainable': artifact_manifest['adapter']['trainable'], 'tensors': len(artifact_manifest['tensors']), 'bytes': artifact_manifest['files'][0]['bytes'], 'sha256': artifact_manifest['files'][0]['sha256'][:16] + '...'})

reloaded = ModNetMattingPipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR, device=pipe.device)
reloaded_test = reloaded.evaluate(test_records)
before = pipe.predict(test_records[:2])['predictions']
after = reloaded.predict(test_records[:2])['predictions']
parity = {'mad_diff': round(abs(reloaded_test['model']['mad'] - adapted_test['model']['mad']), 6), 'metrics_identical': reloaded_test['model'] == adapted_test['model'], 'max_abs_matte_diff': max(float(np.abs(a['alpha'] - b['alpha']).max()) for a, b in zip(before, after))}
print({'reload_parity': parity, 'reloaded_best_epoch': reloaded.adapter['best_epoch']})
assert parity['mad_diff'] < 1e-4 and parity['max_abs_matte_diff'] < 1e-3

result_payload = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model': {**evaluation_report['model'], 'model_license': MODEL_LICENSE, 'device': pipe.device, 'source': pipe.source},
    'provenance': {
        'source_asset': [e for e in MANIFEST['files'] if e['path'] == SOURCE_CKPT_NAME],
        'source_drive_file_id': SOURCE_DRIVE_FILE_ID,
        'upstream_code_commit': UPSTREAM_CODE_COMMIT,
        'pickle_audit_sha256': PICKLE_AUDIT_SHA256,
        'converted': verify_converted(WEIGHTS_DIR)['files'],
        'pickle_unpickled_once_for_conversion': True,
        'served_from_pickle': False,
        'remote_code_executed': False,
        'photographs': [{'id': p['id'], 'url': p['url'], 'bytes': p['bytes'], 'sha256': p['sha256']} for p in PORTRAIT_RECORDS],
        'photograph_license': PORTRAIT_LICENSE,
        'labelled_data': SAMPLE_LABEL_SOURCE,
    },
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'pillow': PIL.__version__, 'numpy': np.__version__},
    'data_source': data_source,
    'comparison': comparison,
    'photograph_drift': photo_drift,
    'artifact': {'dir': str(artifact_dir), 'sha256': artifact_manifest['files'][0]['sha256'], 'bytes': artifact_manifest['files'][0]['bytes']},
    'reload_parity': parity,
}
with open('outputs/modnet_matting_result.json', 'w', encoding='utf-8') as f:
    json.dump(result_payload, f, indent=2)

print('outputs/:')
for path in sorted(Path('outputs').rglob('*')):
    if path.is_file():
        print(f'  - {path.as_posix()} ({path.stat().st_size / 1024:.1f} KB)')

## Interpretation and limits

On 20 held-out drawn portraits the photographic MODNet checkpoint reaches a MAD near 0.08 against constant baselines of 0.34 and 0.66, and a bounded fine-tuning of its matting branches on 48 drawn portraits, selected by validation loss with the frozen model as a candidate, brings it near 0.004. That is the claim: the adaptation contract runs end to end on labelled portrait/alpha pairs, the pickle is audited and converted rather than served, and the artifact that carries the change is about 17 MB and reloads with the same outputs. It is not a claim about matting quality on photographs — the labelled portraits are drawings, chosen because no photographic matting dataset with alpha mattes is both permissively licensed and free of personal-data concerns — and the four photographs are a sanity check without labels, not an evaluation.

The numbers are sample-sanity evidence: one seeded run, 20 test portraits from one renderer, no dispersion estimate, and a domain gap (drawn figures) that makes the gain large by construction. Nothing here measures the model on the PPM-100 benchmark, on video, on group portraits, on hands and objects held in front of the body, or on the hair detail that matting is judged on in practice.

Three things to carry to real data. **The alpha is the contract, and it must belong to the image:** an alpha drawn for another crop, or a binary mask passed off as a matte, is trained on without complaint. **Watch the photographs after adaptation:** fine-tuning on a narrow domain moves the model everywhere, and the frozen-versus-adapted drift on held-out photographs is the early warning. **Read the baselines first:** on a close-up where the subject fills 84 % of the frame the all-foreground matte already scores a MAD of 0.16; only the unknown-band MAD says whether the model resolved the boundary.

Successful execution proves that the recorded repository revision's pipeline modules, carried in this standalone notebook, can acquire and digest-verify a pickled upstream checkpoint, audit and convert it into safetensors without executing anything outside the audited allow-list, build the vendored architecture and load it strictly, render and validate labelled portraits, fetch digest-pinned photographs, execute bounded fine-tuning, evaluate against constant baselines and the frozen model on held-out portraits, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, production fitness, or matting skill on photographs beyond the checks shown.

**Optional experiments (they do not affect the default path):** set `TRAINABLE = 'full'` and compare the photograph drift; raise `EPOCHS` and watch the validation loss drift; try `LEARNING_RATE = 5e-5`; or bring your own labelled portraits through BYOD and read the baselines before the adapted number.

## References

- Repository README: https://github.com/kurtvalcorza/modnet-matting-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/modnet-matting-pipeline/blob/main/MODEL_CARD.md
- Weights, provenance and conversion notes: https://github.com/kurtvalcorza/modnet-matting-pipeline/blob/main/docs/WEIGHTS.md
- Hugging Face mirror of the checkpoint: https://huggingface.co/XM5354/Modnet_models (revision `71aca6d04ed0267b4b12bde776868f2b9fb1d06f`; byte-identical to the authors' Google Drive release)
- Upstream repository (code, models and demos, Apache-2.0): https://github.com/ZHKKKe/MODNet
- Ke, Z., Sun, J., Li, K., Yan, Q., Lau, R. W. H. (2022). MODNet: Real-Time Trimap-Free Portrait Matting via Objective Decomposition. AAAI 2022. arXiv:2011.11961: https://arxiv.org/abs/2011.11961
- Photographs: CC0 Pixabay portraits re-hosted on Wikimedia Commons (URLs pinned in `samples.py`)
- DIMER Notebook Specification 2.0 and Model Card Specification 1.1 (fleet specs in the ml-worker repository)